# CAFA-6 Protein Function Prediction (GO term prediction)

This notebook trains a strong **ESM2 embedding + kNN label transfer** model (a common top-performing approach for CAFA-style problems) and evaluates it on a **held-out 20% test split** from the training data.

Because the Kaggle competition no longer accepts submissions, the original test set (`Test/testsuperset.fasta`) is not used. Instead, the labelled training data is split **80 % train / 20 % test** with two fixed held-out benchmarks: a primary **homology-aware** split for tuning and a secondary **taxon-stratified** split for reporting.

## What you get
- Fast, competitive baseline: ESM2 embeddings (frozen) + cosine kNN over train proteins
- Two fixed 80:20 benchmarks: homology-aware primary split and taxon-stratified secondary split
- GO DAG score propagation: parent score = max child score
- Test-set evaluation with **IA-weighted precision, recall, and max-F** per ontology (MFO, BPO, CCO)

**Default paths assume a local folder** `kaggle/input/cafa-6-protein-function-prediction/` containing the training data.


In [1]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq ncbi-blast+
if ! apt-get install -y -qq diamond-aligner; then
echo "diamond-aligner not available from apt, downloading binary..."
wget -qO /tmp/diamond.tar.gz https://github.com/bbuchfink/diamond/releases/latest/download/diamond-linux64.tar.gz
tar -xzf /tmp/diamond.tar.gz -C /tmp
install -m 755 /tmp/diamond /usr/local/bin/diamond
fi
diamond --version
blastp -version
makeblastdb -version

Couldn't find program: 'bash'


In [2]:
import shutil
print("diamond:", shutil.which("diamond"))
print("blastp:", shutil.which("blastp"))
print("makeblastdb:", shutil.which("makeblastdb"))

diamond: None
blastp: None
makeblastdb: None


In [3]:
!pip install biopython obonet

In [4]:
import random
from Bio import SeqIO

import networkx as nx
import obonet

from sklearn.model_selection import GroupShuffleSplit

In [ ]:
from __future__ import annotations

import json
import logging
import math
import pickle
import random
import shutil
import subprocess
import tempfile
import time
import warnings
from collections import defaultdict
from itertools import combinations
from multiprocessing import Pool
from pathlib import Path
from typing import Any

import faiss
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.special import expit
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import KFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ONTOLOGIES = ["MFO", "BPO", "CCO"]

# Type aliases
TermDict = dict[str, float]
RowScore = dict[str, float]


def is_kaggle() -> bool:
    import os

    return os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""


def load_gene_ontology(obo_path: str | Path) -> tuple[dict[str, str], dict[str, set[str]], dict[str, list[str]]]:
    """
    Parses go-basic.obo. Returns:
      - term_to_ont (dict): term_id -> 'MFO'/'BPO'/'CCO'
      - ancestors_map (dict): term_id -> set of all ancestor term_ids
      - children_map (dict): term_id -> list of immediate child term_ids
    """
    import networkx as nx
    import obonet

    print(f"Loading ontology from {obo_path}...")
    graph = obonet.read_obo(str(obo_path))
    assert isinstance(graph, nx.MultiDiGraph)

    ont_map = {
        "molecular_function": "MFO",
        "biological_process": "BPO",
        "cellular_component": "CCO",
    }
    term_to_ont = {}
    for node, data in graph.nodes(data=True):
        ns = data.get("namespace")
        if ns in ont_map:
            term_to_ont[node] = ont_map[ns]

    # obonet edges point from child -> parent via 'is_a'
    ancestors_map: dict[str, set[str]] = {}
    for node in graph.nodes():
        asts = nx.descendants(graph, node)
        # only keep terms we also track an ontology for
        asts = {a for a in asts if a in term_to_ont}
        ancestors_map[node] = asts

    children_map: dict[str, list[str]] = defaultdict(list)
    for u, v, _key, data in graph.edges(data=True, keys=True):
        # u is child, v is parent.
        # sometimes obonet returns multiple edges between same nodes
        children_map[v].append(u)

    print(f"  Terms loaded: {len(term_to_ont)}")
    return term_to_ont, ancestors_map, dict(children_map)


def propagate_annotations(
    prot_to_terms: dict[str, list[str]], ancestors_map: dict[str, set[str]]
) -> dict[str, list[str]]:
    """Propagate true annotations UP the tree (add all ancestors)."""
    out = {}
    for prot, terms in prot_to_terms.items():
        expanded = set(terms)
        for t in terms:
            expanded.update(ancestors_map.get(t, set()))
        out[prot] = list(expanded)
    return out


def propagate_predictions(
    preds: list[RowScore], children_map: dict[str, list[str]]
) -> list[RowScore]:
    """
    Propagate predictions UP the tree: parent = max(parent, children).
    Works recursively or topologically.
    """
    print("Propagating predictions up the ontology graph...")
    # We do a memoized DFS from roots or just iterate.
    # Since we only want to update parents if a child has a higher score.
    # A generic topological sort is safer.
    # Or just recursive compute:
    out = []
    
    # Build list of all valid terms that have parents/children mapped
    all_known = set(children_map.keys())
    for v in children_map.values():
        all_known.update(v)

    for i, row in enumerate(tqdm(preds)):
        # dict mapping term -> float
        # We need a function that evaluates max(self, max(children))
        # Since graph is DAG, we can use memo dictionary.
        memo = {}
        
        def get_score(t: str) -> float:
            if t in memo:
                return memo[t]
            base = row.get(t, 0.0)
            children = children_map.get(t, [])
            if not children:
                memo[t] = base
                return base
            c_scores = [get_score(c) for c in children if c in all_known]
            c_max = max(c_scores) if c_scores else 0.0
            ans = max(base, c_max)
            memo[t] = ans
            return ans

        new_row = {}
        for t in row.keys():
            sc = get_score(t)
            if sc > 1e-6:
                new_row[t] = sc
        out.append(new_row)
    return out


# -----------------------------
# FAISS k-NN Search
# -----------------------------
def build_faiss_index(emb: np.ndarray, n_threads: int = 4) -> faiss.Index:
    faiss.omp_set_num_threads(n_threads)
    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)

    # Normalize vectors for Cosine Similarity (Inner Product = Cosine if L2=1)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    emb_norm = emb / norms

    index.add(emb_norm.astype(np.float32))
    return index


def search_faiss_index(query_emb: np.ndarray, index: faiss.Index, k: int) -> tuple[np.ndarray, np.ndarray]:
    norms = np.linalg.norm(query_emb, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    query_norm = query_emb / norms
    dists, idxs = index.search(query_norm.astype(np.float32), k)
    return dists, idxs


# -----------------------------
# Heuristic Ensembling (k-NN / IA-weighted)
# -----------------------------
def knn_label_transfer(
    query_emb: np.ndarray,
    train_emb: np.ndarray,
    train_term_lists: list[list[str]],
    index: faiss.Index | None = None,
    k: int = 500,
    top_terms_pre_prop: int = 1000,
    batch_size: int = 2048,
    use_rank_weights: bool = True,
    use_ia_weights: bool = True,
    sqrt_ia_map: dict[str, float] | None = None,
) -> list[RowScore]:
    """
    Transfers labels from k nearest neighbors to query sequences.
    Returns a list of dicts (term -> score).
    """
    if index is None:
        index = build_faiss_index(train_emb)

    n_queries = query_emb.shape[0]
    out = []

    for start in tqdm(range(0, n_queries, batch_size), desc="k-NN transfer"):
        end = min(start + batch_size, n_queries)
        q_batch = query_emb[start:end]
        dists, idxs = search_faiss_index(q_batch, index, k)

        for i in range(end - start):
            term_scores_raw = defaultdict(float)
            term_weight_sum = defaultdict(float)

            # d corresponds to cosine sim (since normalized IP)
            for rank, (nbr_idx, d) in enumerate(zip(idxs[i], dists[i])):
                if nbr_idx < 0:
                    continue
                w = float(d)
                if use_rank_weights:
                    w *= 1.0 / (rank + 1.0)
                
                for t in train_term_lists[nbr_idx]:
                    ia_w = 1.0
                    if use_ia_weights and sqrt_ia_map is not None:
                        ia_w = sqrt_ia_map.get(t, 1.0)
                    
                    term_scores_raw[t] += w * ia_w
                    term_weight_sum[t] += ia_w

            # Normalize and keep top K
            row_dict = {}
            for t, v in term_scores_raw.items():
                if term_weight_sum[t] > 0:
                    row_dict[t] = v / term_weight_sum[t]
            
            # Sort and truncate before propagation to save memory
            sorted_terms = sorted(row_dict.items(), key=lambda cb: cb[1], reverse=True)[:top_terms_pre_prop]
            out.append(dict(sorted_terms))

    return out


# -----------------------------
# Sequence-Similarity Transfer (DIAMOND / BLAST)
# -----------------------------
def write_fasta(ids: list[str], seqs: dict[str, str], path: Path) -> None:
    with open(path, "w") as f:
        for pid in ids:
            if pid in seqs:
                f.write(f">{pid}\n{seqs[pid]}\n")


def _parse_detailed_hits(tsv_path: Path, top_k: int) -> dict[str, list[dict]]:
    hits = defaultdict(list)
    if not tsv_path.exists():
        return hits
    
    with open(tsv_path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.strip().split("\t")
            qseqid = parts[0]
            if len(hits[qseqid]) >= top_k:
                continue

            sseqid = parts[1]
            bitscore = float(parts[2])
            evalue = float(parts[3])
            pident = float(parts[4])
            qcovhsp = float(parts[5]) if len(parts) > 5 else 0.0
            hits[qseqid].append({
                "target": sseqid,
                "bitscore": bitscore,
                "evalue": evalue,
                "pident": pident,
                "qcov": qcovhsp,
            })
    return dict(hits)


def run_diamond_blastp(
    train_ids: list[str],
    test_ids: list[str],
    train_seqs: dict[str, str],
    test_seqs: dict[str, str],
    *,
    top_k: int,
    evalue: float,
    diamond_path: str = "diamond",
) -> dict[str, list[dict]] | None:
    exe = diamond_path if diamond_path else shutil.which("diamond")
    if exe is None:
        print("  diamond not found in PATH; skipping...")
        return None

    tmp = Path(tempfile.mkdtemp())
    try:
        db_fasta = tmp / "train.fasta"
        q_fasta  = tmp / "query.fasta"
        db_path  = tmp / "traindb.dmnd"
        out_tsv  = tmp / "hits.tsv"

        write_fasta(train_ids, train_seqs, db_fasta)
        write_fasta(test_ids, test_seqs, q_fasta)

        subprocess.run(
            [exe, "makedb", "--in", str(db_fasta), "-d", str(db_path), "--quiet"],
            check=True,
            capture_output=True,
            timeout=1800,
        )
        subprocess.run(
            [
                exe,
                "blastp",
                "-q",
                str(q_fasta),
                "-d",
                str(db_path),
                "-o",
                str(out_tsv),
                "-f",
                "6",
                "qseqid",
                "sseqid",
                "bitscore",
                "evalue",
                "pident",
                "qcovhsp",
                "length",
                "qlen",
                "--max-target-seqs",
                str(top_k),
                "--evalue",
                str(evalue),
                "--threads",
                "4",
                "--quiet",
            ],
            check=True,
            capture_output=True,
            timeout=24 * 3600,
        )
        hits = _parse_detailed_hits(out_tsv, top_k=top_k)
        print(
            f"  DIAMOND: {sum(len(v) for v in hits.values())} hits for "        
            f"{len(hits)}/{len(test_ids)} query proteins"
        )
        return hits
    except Exception as exc:
        print(f"  DIAMOND failed ({exc}); trying next method...")
        return None
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def run_blastp(
    train_ids: list[str],
    test_ids: list[str],
    train_seqs: dict[str, str],
    test_seqs: dict[str, str],
    *,
    top_k: int,
    evalue: float,
) -> dict[str, list[dict]] | None:
    if shutil.which("blastp") is None or shutil.which("makeblastdb") is None:   
        print("  blastp not found; skipping...")
        return None

    tmp = Path(tempfile.mkdtemp())
    try:
        db_fasta = tmp / "train.fasta"
        q_fasta = tmp / "query.fasta"
        db_path = tmp / "traindb"
        out_tsv = tmp / "hits.tsv"

        write_fasta(train_ids, train_seqs, db_fasta)
        write_fasta(test_ids, test_seqs, q_fasta)

        subprocess.run(
            ["makeblastdb", "-in", str(db_fasta), "-dbtype", "prot", "-out", str(db_path)],
            check=True,
            capture_output=True,
            timeout=1800,
        )
        subprocess.run(
            [
                "blastp",
                "-query",
                str(q_fasta),
                "-db",
                str(db_path),
                "-outfmt",
                "6 qseqid sseqid bitscore evalue pident qcovhsp length qlen",   
                "-max_target_seqs",
                str(top_k),
                "-evalue",
                str(evalue),
                "-num_threads",
                "4",
                "-out",
                str(out_tsv),
            ],
            check=True,
            capture_output=True,
            timeout=24 * 3600,
        )
        hits = _parse_detailed_hits(out_tsv, top_k=top_k)
        print(
            f"  BLAST: {sum(len(v) for v in hits.values())} hits for "
            f"{len(hits)}/{len(test_ids)} query proteins"
        )
        return hits
    except Exception as exc:
        print(f"  BLASTP failed ({exc}); falling back to k-mer...")
        return None
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def run_kmer_similarity(
    train_ids: list[str],
    test_ids: list[str],
    train_seqs: dict[str, str],
    test_seqs: dict[str, str],
    *,
    k3: int = 3,
    top_k: int,
) -> dict[str, list[dict]]:
    """Fallback TF-IDF k-mer search."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.neighbors import NearestNeighbors

    print(f"  Running k-mer ({k3}-mer) similarity fallback...")

    def get_kmers(seq: str) -> str:
        return " ".join([seq[i : i + k3] for i in range(len(seq) - k3 + 1)])

    train_c = [(pid, get_kmers(train_seqs[pid])) for pid in train_ids if pid in train_seqs]
    test_c = [(pid, get_kmers(test_seqs[pid])) for pid in test_ids if pid in test_seqs]

    if not train_c or not test_c:
        return {}

    train_pids, train_docs = zip(*train_c)
    test_pids, test_docs = zip(*test_c)

    vec = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=2)
    X_train = vec.fit_transform(train_docs)
    X_test = vec.transform(test_docs)

    nn_sz = min(top_k, len(train_pids))
    nbrs = NearestNeighbors(n_neighbors=nn_sz, metric="cosine", n_jobs=-1)
    nbrs.fit(X_train)

    dists, indices = nbrs.kneighbors(X_test)
    # Cosine distance to similarity
    sims = 1.0 - dists

    hits = defaultdict(list)
    for i, pid in enumerate(test_pids):
        for j, rank in enumerate(indices[i]):
            s = float(sims[i, j])
            if s > 1e-4:
                hits[pid].append({
                    "target": train_pids[rank],
                    "bitscore": s * 100.0,
                    "evalue": 1e-10 if s > 0.9 else (1.0 - s),
                    "pident": s * 100.0,
                    "qcov": 100.0,
                })
    return dict(hits)


def compute_seq_sim_hits(
    train_ids: list[str],
    test_ids: list[str],
    train_seqs: dict[str, str],
    test_seqs: dict[str, str],
    *,
    top_k: int = 10,
    evalue: float = 1e-3,
    diamond_path: str = "",
) -> dict[str, list[dict]]:
    print("Computing sequence similarity mappings...")
    hits = run_diamond_blastp(
        train_ids, test_ids, train_seqs, test_seqs, top_k=top_k, evalue=evalue, diamond_path=diamond_path
    )
    if hits is None:
        hits = run_blastp(train_ids, test_ids, train_seqs, test_seqs, top_k=top_k, evalue=evalue)
    if hits is None:
        hits = run_kmer_similarity(train_ids, test_ids, train_seqs, test_seqs, top_k=top_k)
    return dict(hits) if hits else {}


def seq_sim_label_transfer(
    test_ids: list[str],
    hits: dict[str, list[dict]],
    *,
    prot_to_terms: dict[str, list[str]],
    sqrt_ia_map: dict[str, float] | None,
    use_ia_weighted_knn: bool,
) -> tuple[list[RowScore], dict[str, dict[str, float]]]:
    """
    Converts hits -> sequence similarity-based label predictions.
    Returns:
      - row_scores: list of dicts {term -> prob}
      - seq_meta: dict indicating confidence/pident metrics per test_id
    """
    out = []
    seq_meta = {}

    for pid in test_ids:
        pid_hits = hits.get(pid, [])
        if not pid_hits:
            out.append({})
            seq_meta[pid] = {"max_pident": 0.0, "seq_alpha": 0.0}
            continue

        term_scores_raw = defaultdict(float)
        term_weight_sum = defaultdict(float)

        max_pident = 0.0
        for h in pid_hits:
            t_id = h["target"]
            # Convert evalue/bitscore to a pseudo-probability/weight
            # The smaller the evalue, the higher the weight. Wait, pident is 0..100
            pi = h.get("pident", 0.0) / 100.0
            cov = h.get("qcov", 0.0) / 100.0
            
            # Custom heuristic weight combining match quality
            w = pi * (cov ** 0.5)
            if pi > max_pident:
                max_pident = pi

            for t in prot_to_terms.get(t_id, []):
                ia_w = 1.0
                if use_ia_weighted_knn and sqrt_ia_map is not None:
                    ia_w = sqrt_ia_map.get(t, 1.0)
                term_scores_raw[t] += w * ia_w
                term_weight_sum[t] += ia_w

        row_dict = {}
        for t, v in term_scores_raw.items():
            if term_weight_sum[t] > 0:
                row_dict[t] = v / term_weight_sum[t]
        
        # Determine how strictly we want to blend this score (seq_alpha).
        # We cap alpha around 0.8 so we don't totally overwrite ESM2.
        seq_alpha = float(np.clip(max_pident * 1.2, 0.0, 0.8))

        out.append(row_dict)
        seq_meta[pid] = {"max_pident": float(max_pident), "seq_alpha": seq_alpha}

    return out, seq_meta


# -----------------------------
# Deep Supervised Head (2-Layer MLP)
# -----------------------------
def _ranking_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Soft pairwise ranking loss to separate 1s from 0s."""
    pos_mask = targets == 1.0
    neg_mask = targets == 0.0
    if not pos_mask.any() or not neg_mask.any():
        return torch.tensor(0.0, device=logits.device)
    
    # Means per row
    pos_scores = logits.masked_fill(~pos_mask, -1e9)
    neg_scores = logits.masked_fill(~neg_mask, 1e9)
    
    # We want pos_scores > neg_scores -> margin
    # Simplest formulation valid for batch:
    margin = 1.0
    # Average positive score per row, average negative score per row
    avg_pos = pos_scores.mean(dim=1)
    avg_neg = neg_scores.mean(dim=1)
    loss = F.relu(margin - (avg_pos - avg_neg)).mean()
    return loss

class OntologyMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int, dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = F.relu(x)
        x = self.drop(x)
        return self.fc2(x)


def build_supervised_terms(
    train_terms_df: pd.DataFrame,
    ia_map: dict[str, float],
    n_per_ont: dict[str, int] | int,
) -> dict[str, list[str]]:
    out = {}
    for ontology in ONTOLOGIES:
        n = n_per_ont[ontology] if isinstance(n_per_ont, dict) else int(n_per_ont)
        df = train_terms_df[train_terms_df["ontology"] == ontology].copy()      
        if df.empty:
            out[ontology] = []
            continue
        counts = df["term"].value_counts().rename("freq")
        score_df = counts.to_frame()
        score_df["ia"] = score_df.index.map(lambda term: float(ia_map.get(term, 0.0)))
        score_df["priority"] = np.sqrt(score_df["freq"].clip(lower=1)) * score_df["ia"].clip(lower=1e-6)
        out[ontology] = score_df.sort_values(["priority", "freq"], ascending=False).head(n).index.tolist()
    return out


def _expand_terms_for_ontology(
    pid: str,
    *,
    prot_to_terms: dict[str, list[str]],
    term_to_ont: dict[str, str],
    ontology: str,
    ancestors_map: dict[str, set[str]],
    propagate: bool = True,
) -> list[str]:
    direct = [t for t in prot_to_terms.get(pid, []) if term_to_ont.get(t) == ontology]
    if not propagate:
        return direct
    expanded = set(direct)
    for t in direct:
        expanded.update(ancestors_map.get(t, set()))
    return [t for t in expanded if term_to_ont.get(t) == ontology]


def fit_or_load_supervised_head(
    *,
    cache_path: Path,
    train_emb: np.ndarray,
    train_ids: list[str],
    train_terms_df: pd.DataFrame,
    prot_to_terms: dict[str, list[str]],
    term_to_ont: dict[str, str],
    ancestors_map: dict[str, set[str]],
    ia_map: dict[str, float],
    n_per_ont: dict[str, int],
    hidden_dims: dict[str, int],
    device: str,
    seed: int,
    use_propagated_labels: bool,
    dropout: float = 0.3,
    epochs: int = 6,
    batch_size: int = 256,
    learning_rate: float = 3e-4,
) -> dict | None:
    if cache_path.exists():
        print("Loading supervised head from cache:", cache_path)
        return joblib.load(cache_path)

    torch.manual_seed(seed)
    np.random.seed(seed)

    train_id_set = set(train_ids)
    train_terms_df = train_terms_df[train_terms_df["protein"].isin(train_id_set)].copy()
    sup_terms = build_supervised_terms(train_terms_df, ia_map, n_per_ont)       
    payload = {
        "kind": "torch_mlp",
        "sup_terms": sup_terms,
        "models": {},
    }
    input_dim = int(train_emb.shape[1])
    x_train = torch.tensor(train_emb, dtype=torch.float32)

    for ontology in ONTOLOGIES:
        classes = sup_terms.get(ontology, [])
        if not classes:
            payload["models"][ontology] = None
            continue

        class_to_idx = {term: idx for idx, term in enumerate(classes)}
        y_dense = np.zeros((len(train_ids), len(classes)), dtype=np.float32)    
        for row_idx, pid in enumerate(train_ids):
            row_terms = _expand_terms_for_ontology(
                pid,
                prot_to_terms=prot_to_terms,
                term_to_ont=term_to_ont,
                ontology=ontology,
                ancestors_map=ancestors_map,
                propagate=use_propagated_labels,
            )
            for term in set(row_terms):
                col_idx = class_to_idx.get(term)
                if col_idx is not None:
                    y_dense[row_idx, col_idx] = 1.0

        prevalence = y_dense.mean(axis=0)
        prevalence = np.clip(prevalence, 1e-5, 1.0 - 1e-5)
        pos_weight = np.clip((1.0 - prevalence) / prevalence, 1.0, 50.0).astype(np.float32)

        # IA-Weighted Loss integration
        ia_weights = np.array([ia_map.get(term, 1.0) for term in classes], dtype=np.float32)
        ia_weights = np.clip(ia_weights, 0.1, 10.0)
        weight_tensor = torch.tensor(ia_weights, dtype=torch.float32, device=device)

        model = OntologyMLP(input_dim, int(hidden_dims[ontology]), y_dense.shape[1], dropout=dropout).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        y_tensor = torch.tensor(y_dense, dtype=torch.float32)
        pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32, device=device)

        for _ in tqdm(range(epochs), desc=f"Train MLP {ontology}"):
            permutation = np.random.permutation(x_train.shape[0])
            for start in range(0, x_train.shape[0], batch_size):
                batch_idx = permutation[start : start + batch_size]
                xb = x_train[batch_idx].to(device)
                yb = y_tensor[batch_idx].to(device)
                logits = model(xb)
                loss_bce = F.binary_cross_entropy_with_logits(logits, yb, pos_weight=pos_weight_tensor, weight=weight_tensor)
                loss_rank = _ranking_loss(logits, yb)
                loss = loss_bce + 0.2 * loss_rank
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        model = model.to("cpu")
        payload["models"][ontology] = {
            "state_dict": model.state_dict(),
            "input_dim": input_dim,
            "hidden_dim": int(hidden_dims[ontology]),
            "output_dim": y_dense.shape[1],
            "dropout": dropout,
            "classes": list(classes),
            "top_k": min(128, y_dense.shape[1]),
        }
        density = float(y_dense.mean()) if y_dense.size else 0.0
        print(f"  {ontology}: {y_dense.shape[1]} labels, density {density:.4f}")

    joblib.dump(payload, cache_path)
    print("Saved supervised head cache:", cache_path)
    return payload


def supervised_scores(payload: dict | None, emb: np.ndarray, *, device: str) -> list[dict]:
    if payload is None:
        return [dict() for _ in range(emb.shape[0])]

    out = [defaultdict(float) for _ in range(emb.shape[0])]
    x = torch.tensor(emb, dtype=torch.float32)

    for ontology in ONTOLOGIES:
        spec = payload["models"].get(ontology)
        if spec is None:
            continue
        model = OntologyMLP(spec["input_dim"], spec["hidden_dim"], spec["output_dim"], dropout=spec["dropout"])
        model.load_state_dict(spec["state_dict"])
        model.to(device)
        model.eval()

        with torch.no_grad():
            batch_size = 512
            logits_list = []
            for start in range(0, x.shape[0], batch_size):
                end = start + batch_size
                xb = x[start:end].to(device)
                lb = model(xb)
                logits_list.append(lb.cpu())
            all_logits = torch.cat(logits_list, dim=0).numpy()

        probs = expit(all_logits)
        classes = spec["classes"]
        top_k_n = spec.get("top_k", 128)

        for i in range(emb.shape[0]):
            row_p = probs[i]
            # Use argpartition for fast top_k
            top_idx = np.argpartition(-row_p, kth=top_k_n - 1)[:top_k_n]
            for j in top_idx:
                if row_p[j] > 1e-4:
                    out[i][classes[j]] = float(row_p[j])

    return [dict(d) for d in out]


def blend_score_dicts(d1: RowScore, d2: RowScore, w1: float) -> RowScore:
    """Takes dicts d1 and d2. Result = w1*d1 + (1-w1)*d2."""
    out = defaultdict(float)
    w2 = 1.0 - w1
    for k, v in d1.items():
        out[k] += v * w1
    for k, v in d2.items():
        out[k] += v * w2
    return dict(out)


# -----------------------------
# Logistic Fusion Stack
# -----------------------------
def build_term_depth_map(ancestors_map: dict[str, set[str]], term_to_ont: dict[str, str]) -> dict[str, int]:
    depth_map = {}
    for term in ancestors_map.keys():
        depth_map[term] = len(ancestors_map.get(term, set()))
    return depth_map

def extract_stack_features(
    knn_score: float,
    sup_score: float,
    seq_score: float,
    term: str,
    term_to_ont: dict[str, str],
    ia_map: dict[str, float],
    depth_map: dict[str, int],
    meta: dict[str, Any],
    taxon_meta: dict[str, Any],
) -> np.ndarray:
    
    ont = term_to_ont.get(term, "MFO")
    ia = ia_map.get(term, 1.0)
    depth = depth_map.get(term, 1)

    max_pident = meta.get("max_pident", 0.0)
    seq_alpha = meta.get("seq_alpha", 0.0)
    taxon_alpha = taxon_meta.get("taxon_alpha", 0.0)

    # 10 features total per term-prediction
    return np.array([
        knn_score,
        sup_score,
        seq_score,
        max_pident,
        seq_alpha,
        taxon_alpha,
        ia,
        depth,
        float(ont == "MFO"),
        float(ont == "BPO"),
    ], dtype=np.float32)


def fit_ontology_stackers(
    target_ids: list[str],
    knn_scores: list[RowScore],
    sup_scores: list[RowScore],
    seq_scores: list[RowScore],
    true_by_prot: dict[str, set[str]],
    term_to_ont: dict[str, str],
    ia_map: dict[str, float],
    depth_map: dict[str, int],
    seq_meta_by_pid: dict[str, dict[str, float]],
    taxon_meta_by_pid: dict[str, dict[str, float]],
) -> dict[str, BaseEstimator]:
    """Train logistic regression stackers to optimally combine knn, sup, and seq signals."""
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression

    print("Extracting features for logistic fusion stack...")
    
    datasets = {o: {"X": [], "y": []} for o in ONTOLOGIES}

    for i, pid in enumerate(target_ids):
        knn = knn_scores[i]
        sup = sup_scores[i] if sup_scores else {}
        seq = seq_scores[i] if seq_scores else {}
        meta = seq_meta_by_pid.get(pid, {})
        taxon = taxon_meta_by_pid.get(pid, {})

        all_terms = set(knn.keys()) | set(sup.keys()) | set(seq.keys())
        true_terms = true_by_prot.get(pid, set())

        for t in all_terms:
            o = term_to_ont.get(t)
            if o not in datasets:
                continue

            ks = knn.get(t, 0.0)
            ss = sup.get(t, 0.0)
            qs = seq.get(t, 0.0)

            # Skip trivial 0s
            if ks < 1e-4 and ss < 1e-4 and qs < 1e-4:
                continue

            feats = extract_stack_features(ks, ss, qs, t, term_to_ont, ia_map, depth_map, meta, taxon)
            label = 1 if t in true_terms else 0

            datasets[o]["X"].append(feats)
            datasets[o]["y"].append(label)

    models = {}
    for o in ONTOLOGIES:
        X = np.array(datasets[o]["X"])
        y = np.array(datasets[o]["y"])
        if len(np.unique(y)) > 1:
            print(f"  Training stacker for {o} on {X.shape[0]} samples (pos density {y.mean():.4f})")
            clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=200, random_state=42))
            clf.fit(X, y)
            models[o] = clf
        else:
            print(f"  Skipping {o} stacker due to uniform classes.")

    return models


def stack_ontology_scores(
    target_ids: list[str],
    knn_scores: list[RowScore],
    sup_scores: list[RowScore],
    seq_scores: list[RowScore],
    term_to_ont: dict[str, str],
    stackers: dict[str, BaseEstimator],
    ia_map: dict[str, float],
    depth_map: dict[str, int],
    seq_meta_by_pid: dict[str, dict[str, float]],
    taxon_meta_by_pid: dict[str, dict[str, float]],
) -> list[RowScore]:
    
    out = []
    
    for i, pid in enumerate(tqdm(target_ids, desc="Fusing stack scores")):
        knn = knn_scores[i]
        sup = sup_scores[i] if sup_scores else {}
        seq = seq_scores[i] if seq_scores else {}
        meta = seq_meta_by_pid.get(pid, {})
        taxon = taxon_meta_by_pid.get(pid, {})

        all_terms = set(knn.keys()) | set(sup.keys()) | set(seq.keys())
        row_out = {}
        
        # Batch inference per ontology
        groups = defaultdict(list)
        term_groups = defaultdict(list)
        for t in all_terms:
            o = term_to_ont.get(t)
            if o in stackers:
                ks = knn.get(t, 0.0)
                ss = sup.get(t, 0.0)
                qs = seq.get(t, 0.0)
                feats = extract_stack_features(ks, ss, qs, t, term_to_ont, ia_map, depth_map, meta, taxon)
                groups[o].append(feats)
                term_groups[o].append(t)
            else:
                # fallback averaging 
                ks = knn.get(t, 0.0)
                ss = sup.get(t, 0.0)
                row_out[t] = 0.5 * ks + 0.5 * ss

        for o in groups:
            X = np.array(groups[o])
            clf = stackers[o]
            if hasattr(clf, "predict_proba"):
                probs = clf.predict_proba(X)[:, 1]
            else:
                probs = clf.decision_function(X)
                probs = expit(probs)
            
            for t, p in zip(term_groups[o], probs):
                row_out[t] = float(p)
                
        out.append(row_out)
        
    return out


# -----------------------------
# Calibration
# -----------------------------
def fit_ontology_calibrators(
    pred_map: dict[str, RowScore],
    true_by_prot: dict[str, set[str]],
    term_to_ont: dict[str, str],
    mode: str = 'platt',
) -> dict[str, Any]:
    print(f"Fitting {mode} calibrators per ontology...")
    from sklearn.isotonic import IsotonicRegression

    # Gather data
    datasets = {o: {"score": [], "y": []} for o in ONTOLOGIES}
    
    for pid, preds in pred_map.items():
        true_set = true_by_prot.get(pid, set())
        for t, sc in preds.items():
            o = term_to_ont.get(t)
            if o:
                datasets[o]["score"].append(sc)
                datasets[o]["y"].append(1 if t in true_set else 0)

    calibrators = {}
    for o in ONTOLOGIES:
        X = np.array(datasets[o]["score"], dtype=float)
        y = np.array(datasets[o]["y"], dtype=int)
        
        if len(np.unique(y)) > 1:
            if mode == 'isotonic':
                iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds='clip')
                iso.fit(X, y)
                calibrators[o] = iso
            elif mode == 'platt':
                clf = LogisticRegression(class_weight='balanced')
                clf.fit(X.reshape(-1, 1), y)
                calibrators[o] = clf
            print(f"  {o} Calibrator ({mode}) fit on {len(X)} samples.")
            
    return calibrators

def calibrate_scores(
    row: RowScore, 
    calibrators: dict[str, Any], 
    term_to_ont: dict[str, str],
) -> RowScore:
    out = {}
    for t, sc in row.items():
        o = term_to_ont.get(t)
        cal = calibrators.get(o)
        if cal:
            try:
                if hasattr(cal, 'predict_proba'):
                    sc = float(cal.predict_proba([[sc]])[0, 1])
                else:
                    sc = float(cal.predict([sc])[0])
            except Exception:
                pass
        out[t] = sc
    return out


def truncate_predictions(row: RowScore, max_terms: int = 1500) -> RowScore:
    items = sorted(row.items(), key=lambda x: x[1], reverse=True)[:max_terms]
    return {k: v for k, v in items if v > 1e-6}


In [6]:
# -----------------------------
# Config
# -----------------------------
SEED = 472
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DATA_DIR = Path('/kaggle/input/cafa-6-protein-function-prediction')

# -----------------------------
# Train / Test split
# -----------------------------
TEST_SPLIT_RATIO = 0.20
SPLIT_MODE       = 'homology_80_20'
REPORT_SPLIT_MODES = ['homology_80_20', 'taxon_80_20']
USE_DEV_SUBSET   = False
DEV_SUBSET_TOTAL = 500

# -----------------------------
# Backbone / embeddings
# -----------------------------
MODEL_NAME        = 'facebook/esm2_t33_650M_UR50D'
USE_PROTT5        = False
PROTT5_MODEL_NAME = 'Rostlab/prot_t5_xl_uniref50'
MAX_AA_PER_CHUNK  = 1000
BATCH_SIZE        = 4 if DEVICE == 'cuda' else 1
POOLING_MODE      = 'last3_concat'

# -----------------------------
# kNN config
# -----------------------------
K_NEIGHBORS = 500
TUNE_K      = False
K_GRID      = [100, 200, 400, 600]

USE_RANK_WEIGHTED_KNN = True   # weight neighbour r by 1/(r+1)
USE_IA_WEIGHTED_KNN   = True   # multiply per-term contribution by sqrt(IA)
MIN_TERM_FREQ         = 3      # drop terms annotating < 3 train proteins

# Taxonomy-aware ensemble
USE_TAXON_ENSEMBLE = True
TAXON_BLEND_ALPHA  = 0.65
MIN_TAXON_TRAIN    = 200

# Option 1 — Bayesian smoothing (population-level trust):
#   α_bayes(t) = N_t / (N_t + λ)
#   Large taxa (Human, E. coli) → α→1. Sparse taxa → α scales toward global.
#   λ=100 means a taxon needs 100+ proteins to get α>0.5.
#
# Option 2 — Similarity-weighted (per-query retrieval confidence):
#   α_sim = S_taxon / (S_taxon + S_global)
#   If a query protein has a 0.95 cosine hit in its own taxon but only 0.30
#   outside, α_sim→0.76, trusting the taxon prediction heavily. If it has no
#   better taxon match than the global pool, α_sim→0.5.
#
# Combined: α = α_sim × α_bayes
#   The Bayesian term acts as a population-size floor so that per-query
#   similarity confidence from a sparse taxon is automatically down-weighted.
USE_DYNAMIC_TAXON_ALPHA = True
TAXON_BAYESIAN_LAMBDA   = 100.0   # smoothing prior for Option 1

# -----------------------------
# Supervised head
# -----------------------------
USE_SUPERVISED_HEAD       = True
SUP_TERMS_PER_ONTOLOGY    = {'MFO': 768, 'BPO': 3072, 'CCO': 768}
SUP_BLEND_WEIGHT          = 0.30
USE_PROPAGATED_SUP_LABELS = True   # propagate GO ancestors into training targets
MLP_HIDDEN_DIMS          = {'MFO': 1024, 'BPO': 2048, 'CCO': 1024}
MLP_EPOCHS               = 6

# BPO coverage floor
BPO_COVERAGE_FLOOR = True
BPO_FLOOR_SCORE    = 0.002

# -----------------------------
# Sequence-similarity signal  (Improvement #2)
# -----------------------------
USE_SEQ_SIM          = True
SEQ_SIM_TOP_K        = 10
SEQ_SIM_EVALUE       = 1e-3
SEQ_SIM_BLEND_WEIGHT = 0.25
DIAMOND_PATH         = ''
HOMOLOGY_SPLIT_EVALUE = 1e-3

# Calibration / fusion
CALIBRATION_MODE = 'platt'
FUSION_MODE      = 'logistic_stack'
USE_PER_PROTEIN_MAX_NORMALIZATION = False

# Post-processing
TOP_TERMS_PER_PROTEIN_BEFORE_PROP = 1000
MAX_TERMS_PER_PROTEIN_FINAL       = 1500
MIN_SCORE                         = 1e-6

# -----------------------------
# Cache layout
# -----------------------------
_BACKBONE  = f"{MODEL_NAME.replace('/', '_')}_{POOLING_MODE}"
_DEV_TAG   = f'dev{DEV_SUBSET_TOTAL}' if USE_DEV_SUBSET else 'full'
_SPLIT_TAG = f'{SPLIT_MODE}_{_DEV_TAG}_seed{SEED}_split{int(TEST_SPLIT_RATIO * 100)}'

_ALGO_TAG = (f'k{K_NEIGHBORS}'
             f'_mf{MIN_TERM_FREQ}'
             f'_rw{int(USE_RANK_WEIGHTED_KNN)}'
             f'_iaw{int(USE_IA_WEIGHTED_KNN)}'
             f'_ss{int(USE_SEQ_SIM)}'
             f'_dta{int(USE_DYNAMIC_TAXON_ALPHA)}'
             f'_cal{CALIBRATION_MODE}'
             f'_fus{FUSION_MODE}')

_bpo_n  = SUP_TERMS_PER_ONTOLOGY['BPO']
_mfo_n  = SUP_TERMS_PER_ONTOLOGY['MFO']
_cco_n  = SUP_TERMS_PER_ONTOLOGY['CCO']
_prop   = 'proplbl' if USE_PROPAGATED_SUP_LABELS else 'dirlbl'
_mlp_tag = f"mlp_mfo{MLP_HIDDEN_DIMS['MFO']}_bpo{MLP_HIDDEN_DIMS['BPO']}_cco{MLP_HIDDEN_DIMS['CCO']}_ep{MLP_EPOCHS}"
_SUP_TAG = f'sup_{_mlp_tag}_mfo{_mfo_n}_bpo{_bpo_n}_cco{_cco_n}_{_prop}'

WORK_DIR = Path('./cafa_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DEV_SUBSET_CACHE_PATH    = WORK_DIR / f'labelled_subset_{_DEV_TAG}_seed{SEED}.json'
TAXON_SPLIT_CACHE_PATH   = WORK_DIR / f'train_test_split_taxon_{_DEV_TAG}_seed{SEED}_split{int(TEST_SPLIT_RATIO * 100)}.json'
HOMOLOGY_SPLIT_CACHE_PATH = WORK_DIR / f"train_test_split_homology_{_DEV_TAG}_e{str(HOMOLOGY_SPLIT_EVALUE).replace('.', 'p')}_seed{SEED}_split{int(TEST_SPLIT_RATIO * 100)}.json"

TRAIN_EMB_PATH      = WORK_DIR / f'train_emb_{_BACKBONE}_{_SPLIT_TAG}.npy'
TRAIN_IDS_PATH      = WORK_DIR / f'train_ids_{_BACKBONE}_{_SPLIT_TAG}.json'
TEST_SPLIT_EMB_PATH = WORK_DIR / f'test_split_emb_{_BACKBONE}_{_SPLIT_TAG}.npy'
TEST_SPLIT_IDS_PATH = WORK_DIR / f'test_split_ids_{_BACKBONE}_{_SPLIT_TAG}.json'

_PT5 = PROTT5_MODEL_NAME.replace('/', '_')
PROTT5_TRAIN_EMB_PATH      = WORK_DIR / f'train_emb_{_PT5}_{_SPLIT_TAG}.npy'
PROTT5_TRAIN_IDS_PATH      = WORK_DIR / f'train_ids_{_PT5}_{_SPLIT_TAG}.json'
PROTT5_TEST_SPLIT_EMB_PATH = WORK_DIR / f'test_split_emb_{_PT5}_{_SPLIT_TAG}.npy'
PROTT5_TEST_SPLIT_IDS_PATH = WORK_DIR / f'test_split_ids_{_PT5}_{_SPLIT_TAG}.json'

SUP_CACHE_PATH        = WORK_DIR / f'sup_head_{_BACKBONE}_{_SPLIT_TAG}_{_SUP_TAG}.joblib'
SEQ_SIM_CACHE_PATH    = WORK_DIR / f"seq_sim_hits_{_SPLIT_TAG}_top{SEQ_SIM_TOP_K}_e{str(SEQ_SIM_EVALUE).replace('.', 'p')}.pkl"
TEST_PREDS_CACHE_PATH = WORK_DIR / f'test_preds_{_BACKBONE}_{_SPLIT_TAG}_{_ALGO_TAG}_{_SUP_TAG}.pkl'

print('Device:', DEVICE)
print('Cache directory:', WORK_DIR.resolve())
print('Split mode:', SPLIT_MODE, '| Report splits:', REPORT_SPLIT_MODES, '| Dev subset:', USE_DEV_SUBSET, '| Pooling:', POOLING_MODE, '| Calibration:', CALIBRATION_MODE, '| Fusion:', FUSION_MODE)
if USE_DEV_SUBSET:
    print(f'Dev subset size: {DEV_SUBSET_TOTAL} proteins -> target split {(1.0 - TEST_SPLIT_RATIO):.0%}/{TEST_SPLIT_RATIO:.0%}')
print('Split tag:', _SPLIT_TAG, '| Algo tag:', _ALGO_TAG, '| Sup tag:', _SUP_TAG)
print('Existing cache files:')
_cache_files = sorted(WORK_DIR.glob('*'))
if _cache_files:
    for f in _cache_files:
        print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')
else:
    print('  (none)')

# Kaggle cache persistence helpers (optional)
def export_cache_archive(out_path: str | Path = '/kaggle/working/cafa_work_cache.tar.gz') -> Path:
    import tarfile

    out_path = Path(out_path)
    with tarfile.open(out_path, 'w:gz') as tar:
        for f in sorted(WORK_DIR.glob('*')):
            if f.is_file():
                tar.add(f, arcname=f.name)
    print(f'Cache archive saved -> {out_path}')
    return out_path

def restore_cache_archive(archive_path: str | Path) -> int:
    import tarfile

    archive_path = Path(archive_path)
    if not archive_path.exists():
        print(f'Cache archive not found: {archive_path}')
        return 0

    restored = 0
    root = WORK_DIR.resolve()
    with tarfile.open(archive_path, 'r:gz') as tar:
        for member in tar.getmembers():
            if member.isfile():
                target = (WORK_DIR / member.name).resolve()
                if not str(target).startswith(str(root)):
                    continue
                tar.extract(member, WORK_DIR)
                restored += 1
    print(f'Restored {restored} files into {WORK_DIR}')
    return restored

print("Cache helpers ready: restore_cache_archive('/kaggle/input/<dataset>/cafa_work_cache.tar.gz')")
print("                     export_cache_archive('/kaggle/working/cafa_work_cache.tar.gz')")

# Optional auto-restore from attached Kaggle datasets.
AUTO_RESTORE_CACHE = True
AUTO_RESTORE_ARCHIVE_NAME = 'cafa_work_cache.tar.gz'
AUTO_RESTORE_ARCHIVE_PATH = ''  # set explicit path to override auto-search

def _find_auto_restore_archive() -> Path | None:
    if AUTO_RESTORE_ARCHIVE_PATH:
        explicit = Path(AUTO_RESTORE_ARCHIVE_PATH)
        if explicit.exists():
            return explicit

    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None

    for dataset_dir in sorted(input_root.iterdir()):
        candidate = dataset_dir / AUTO_RESTORE_ARCHIVE_NAME
        if candidate.exists():
            return candidate
    return None

if AUTO_RESTORE_CACHE:
    _auto_archive = _find_auto_restore_archive()
    if _auto_archive is not None:
        print(f'Auto-restore cache from {_auto_archive}')
        restore_cache_archive(_auto_archive)
    else:
        print(f'Auto-restore skipped: no {AUTO_RESTORE_ARCHIVE_NAME} under /kaggle/input/*')
else:
    print('Auto-restore disabled (AUTO_RESTORE_CACHE=False)')


Device: cpu
Cache directory: D:\Academic\Level-4 Term-II\CSE 472\project\cafa_work
Split mode: homology_80_20 | Report splits: ['homology_80_20', 'taxon_80_20'] | Dev subset: False | Pooling: last3_concat | Calibration: platt | Fusion: logistic_stack
Split tag: homology_80_20_full_seed472_split20 | Algo tag: k500_mf3_rw1_iaw1_ss1_dta1_calplatt_fuslogistic_stack | Sup tag: sup_mlp_mfo1024_bpo2048_cco1024_ep6_mfo768_bpo3072_cco768_proplbl
Existing cache files:
  ancestors_cache.json  (5636 KB)
  seq_sim_hits_seed472_split20_top10.pkl  (2710 KB)
  sup_head_facebook_esm2_t12_35M_UR50D_seed472_split20_512.joblib  (3868 KB)
  sup_head_facebook_esm2_t12_35M_UR50D_seed472_split20_sup_mfo512_bpo2048_cco512.joblib  (7735 KB)
  sup_head_facebook_esm2_t12_35M_UR50D_seed472_split20_sup_mfo768_bpo3072_cco768.joblib  (11602 KB)
  sup_head_facebook_esm2_t12_35M_UR50D_seed472_split20_sup_mfo768_bpo3072_cco768_proplbl.joblib  (11602 KB)
  test_preds_facebook_esm2_t12_35M_UR50D_seed472_split20.pkl  (3377

## Load data
We read:
- Train sequences: `Train/train_sequences.fasta`
- Train terms: `Train/train_terms.tsv`
- Train taxonomy: `Train/train_taxonomy.tsv` (used for the 80:20 group split)
- IA weights: `IA.tsv`
- GO DAG: `Train/go-basic.obo`

The original `Test/` folder is **not used** — true labels for those proteins are unavailable.
A held-out 20 % test split is carved out from the labelled training data instead.


In [7]:
train_fasta      = DATA_DIR / 'Train' / 'train_sequences.fasta'
train_terms_tsv  = DATA_DIR / 'Train' / 'train_terms.tsv'
train_tax_tsv    = DATA_DIR / 'Train' / 'train_taxonomy.tsv'
go_obo           = DATA_DIR / 'Train' / 'go-basic.obo'
ia_tsv           = DATA_DIR / 'IA.tsv'

for p in [train_fasta, train_terms_tsv, train_tax_tsv, go_obo, ia_tsv]:
    assert p.exists(), f'Missing file: {p}'

import re

def _parse_uniprot_id(rec: SeqIO.SeqRecord) -> str:
    header = rec.id
    if '|' in header:
        parts = header.split('|')
        if len(parts) >= 2 and parts[1]:
            return parts[1]
    return header

_TAXON_PATTERNS = [
    re.compile(r'(?:taxon:|taxid:|TaxID=)(\d+)', re.IGNORECASE),
    re.compile(r'\bOX=(\d+)\b'),
]

def _parse_taxon_id(rec: SeqIO.SeqRecord) -> int:
    desc = rec.description or ''
    for pat in _TAXON_PATTERNS:
        m = pat.search(desc)
        if m:
            return int(m.group(1))
    return -1


def read_fasta(path: Path):
    """Returns (seqs, taxon_map). taxon_map values can be -1 if not found in header."""
    seqs = {}
    taxon_map = {}
    for rec in SeqIO.parse(str(path), 'fasta'):
        pid = _parse_uniprot_id(rec)
        seqs[pid] = str(rec.seq)
        taxon_map[pid] = _parse_taxon_id(rec)
    return seqs, taxon_map

train_seqs, train_tax_from_fasta = read_fasta(train_fasta)
print('Total labelled proteins in train FASTA:', len(train_seqs))

train_terms = pd.read_csv(train_terms_tsv, sep='\t', header=None, names=['protein', 'term', 'ontology'])
train_tax   = pd.read_csv(train_tax_tsv,   sep='\t', header=None, names=['protein', 'taxon'])
ia          = pd.read_csv(ia_tsv,          sep='\t', header=None, names=['term', 'ia'])

print(train_terms.head())
print(train_tax.head())
print(ia.head())


AssertionError: Missing file: \kaggle\input\cafa-6-protein-function-prediction\Train\train_sequences.fasta

In [ ]:
# Remap single-letter ontology codes → 3-letter names
_ONT_REMAP = {'C': 'CCO', 'F': 'MFO', 'P': 'BPO'}
train_terms['ontology'] = train_terms['ontology'].replace(_ONT_REMAP)

# Keep only terms that have IA weights
ia_terms = set(ia['term'].tolist())
train_terms = train_terms[train_terms['term'].isin(ia_terms)].reset_index(drop=True)

# Build term lists per ontology
ontologies   = ['BPO', 'CCO', 'MFO']
terms_by_ont = {o: sorted(train_terms.loc[train_terms['ontology'] == o, 'term'].unique().tolist())
                for o in ontologies}
print('Terms per ontology:', {o: len(terms_by_ont[o]) for o in ontologies})

# Map term -> ontology
term_to_ont = {}
for o in ontologies:
    for t in terms_by_ont[o]:
        term_to_ont[t] = o

# Build protein -> list of terms
prot_to_terms = defaultdict(list)
for p, t in zip(train_terms['protein'].values, train_terms['term'].values):
    prot_to_terms[p].append(t)

# Ensure protein ids exist in fasta
missing = [p for p in prot_to_terms.keys() if p not in train_seqs]
print('Train term proteins missing sequences:', len(missing))
if len(missing) > 0:
    train_terms = train_terms[train_terms['protein'].isin(train_seqs.keys())].reset_index(drop=True)
    prot_to_terms = defaultdict(list)
    for p, t in zip(train_terms['protein'].values, train_terms['term'].values):
        prot_to_terms[p].append(t)

print('Total labelled proteins (have ≥1 GO term with IA weight):', len(prot_to_terms))

# ------------------------------------------------------------------
# Min-frequency filter: discard training terms that appear in fewer
# than MIN_TERM_FREQ proteins.  Rare terms add noise to kNN transfer
# and hurt precision without improving recall.
# ------------------------------------------------------------------
from collections import Counter as _TermCounter
_term_freq = _TermCounter(t for terms in prot_to_terms.values() for t in terms)
_rare_terms = {t for t, c in _term_freq.items() if c < MIN_TERM_FREQ}
if _rare_terms:
    _before = sum(len(v) for v in prot_to_terms.values())
    prot_to_terms = defaultdict(list, {
        p: [t for t in ts if t not in _rare_terms]
        for p, ts in prot_to_terms.items()
    })
    _after = sum(len(v) for v in prot_to_terms.values())
    print(f'Min-freq filter (MIN_TERM_FREQ={MIN_TERM_FREQ}): '
          f'removed {len(_rare_terms)} terms, {_before - _after} annotations dropped')

# ------------------------------------------------------------------
# Precompute sqrt(IA) lookup used by IA-weighted kNN scoring.
# sqrt is used instead of raw IA to avoid over-amplifying very
# specific (high-IA) leaf terms at the expense of intermediate terms.
# ------------------------------------------------------------------
_ia_values = dict(zip(ia['term'].values, ia['ia'].values))
_mean_sqrt_ia = float(np.mean([np.sqrt(v) for v in _ia_values.values() if v > 0]))
sqrt_ia_map = {t: float(np.sqrt(_ia_values.get(t, 0.0))) / (_mean_sqrt_ia + 1e-9)
               for t in ia_terms}
print(f'sqrt_ia_map built for {len(sqrt_ia_map)} IA terms (mean sqrt-IA normalised to 1)')


Terms per ontology: {'BPO': 16858, 'CCO': 2651, 'MFO': 6616}
Train term proteins missing sequences: 0
Total labelled proteins (have ≥1 GO term with IA weight): 82404
Min-freq filter (MIN_TERM_FREQ=3): removed 8817 terms, 12257 annotations dropped
sqrt_ia_map built for 40122 IA terms (mean sqrt-IA normalised to 1)


## 80 / 20 Train–Test Split
We split the **labelled proteins** (those present in both `train_sequences.fasta` and `train_terms.tsv`) into a **training set (80 %)** used to build the kNN index and train the supervised head, and a **held-out test set (20 %)** used only for the final evaluation.

Two persistent benchmark definitions are created. `homology_80_20` groups proteins by sequence-connected components before splitting and is the primary tuning benchmark. `taxon_80_20` keeps the older taxon-stratified split as a secondary reporting benchmark.


In [ ]:
# Helper functions for split building
def sample_labelled_subset(all_labelled_ids: list[str], train_tax_map: dict[str, int], *,
                          subset_size: int, seed: int, cache_path=None) -> list[str]:
    """Sample a subset of labelled proteins, balancing by taxon."""
    import math
    all_labelled_ids = sorted(all_labelled_ids)
    subset_size = int(subset_size)
    if subset_size <= 0 or subset_size >= len(all_labelled_ids):
        return all_labelled_ids
    
    if cache_path is not None and cache_path.exists():
        payload = json.loads(cache_path.read_text())
        subset_ids = sorted(payload.get("subset_ids", []))
        if len(subset_ids) == subset_size and set(subset_ids).issubset(set(all_labelled_ids)):
            return subset_ids
    
    grouped = defaultdict(list)
    for pid in all_labelled_ids:
        grouped[int(train_tax_map.get(pid, -1))].append(pid)
    
    total = len(all_labelled_ids)
    expected = {taxon: subset_size * (len(ids) / total) for taxon, ids in grouped.items()}
    allocated = {taxon: min(len(ids), int(math.floor(expected[taxon]))) for taxon, ids in grouped.items()}
    
    remaining = subset_size - sum(allocated.values())
    while remaining > 0:
        candidates = [taxon for taxon, ids in grouped.items() if allocated[taxon] < len(ids)]
        if not candidates:
            break
        candidates.sort(key=lambda t: (expected[t] - allocated[t], len(grouped[t]), -t), reverse=True)
        for taxon in candidates:
            if remaining <= 0:
                break
            if allocated[taxon] < len(grouped[taxon]):
                allocated[taxon] += 1
                remaining -= 1
    
    rng = np.random.default_rng(seed)
    subset_ids = []
    for taxon in sorted(grouped):
        ids = grouped[taxon]
        n_take = allocated[taxon]
        if n_take <= 0:
            continue
        if n_take >= len(ids):
            subset_ids.extend(ids)
        else:
            chosen = rng.choice(np.asarray(ids, dtype=object), size=n_take, replace=False)
            subset_ids.extend(chosen.tolist())
    
    subset_ids = sorted(set(subset_ids))
    if cache_path is not None:
        cache_path.write_text(json.dumps({"subset_size": subset_size, "subset_ids": subset_ids}))
    return subset_ids

def build_taxon_stratified_split(all_labelled_ids: list[str], train_tax_map: dict[str, int], *, 
                                test_size: float, seed: int, cache_path=None) -> tuple[list[str], list[str]]:
    """Build a stratified 80/20 split by taxon."""
    if cache_path is not None and cache_path.exists():
        payload = json.loads(cache_path.read_text())
        train_ids = sorted(payload.get("train_ids", []))
        test_ids = sorted(payload.get("test_ids", []))
        if sorted(train_ids + test_ids) == sorted(all_labelled_ids):
            return train_ids, test_ids
    
    from sklearn.model_selection import train_test_split
    taxon_labels = [int(train_tax_map.get(pid, -1)) for pid in all_labelled_ids]
    counts = Counter(taxon_labels)
    strat_labels = [tx if counts[tx] >= 2 else -1 for tx in taxon_labels]
    
    train_ids, test_ids = train_test_split(list(all_labelled_ids), test_size=test_size, 
                                           random_state=seed, stratify=strat_labels)
    train_ids = sorted(train_ids)
    test_ids = sorted(test_ids)
    
    if cache_path is not None:
        cache_path.write_text(json.dumps({"mode": "taxon_80_20", "train_ids": train_ids, "test_ids": test_ids}))
    return train_ids, test_ids

def build_homology_aware_split(all_labelled_ids: list[str], seq_dict: dict[str, str], train_tax_map: dict[str, int], *,
                              test_size: float, seed: int, cache_path, work_dir, diamond_path='', evalue=1e-3) -> tuple[list[str], list[str], dict]:
    """Build a homology-aware split, returning (train_ids, test_ids, component_map)."""
    all_labelled_ids = sorted(all_labelled_ids)
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    cached_split = _load_cached_split(cache_path, all_labelled_ids)
    component_cache_path = _component_cache_path(cache_path)
    if cached_split is not None and component_cache_path.exists():
        train_ids, test_ids = cached_split
        payload = json.loads(component_cache_path.read_text())
        component_map = {pid: int(payload.get(pid, -1)) for pid in all_labelled_ids}
        print(f'Loaded cached homology split: train={len(train_ids):,} test={len(test_ids):,}')
        return train_ids, test_ids, component_map

    diamond_exe = None
    if diamond_path:
        diamond_exe = diamond_path
    else:
        for candidate in ('diamond', 'diamond.exe'):
            if shutil.which(candidate):
                diamond_exe = candidate
                break

    edges: list[tuple[str, str]] = []
    if diamond_exe:
        print(f'Building homology graph with DIAMOND ({diamond_exe})...')
        try:
            edges = _run_pairwise_diamond_edges(
                all_labelled_ids,
                seq_dict,
                evalue=evalue,
                diamond_exe=diamond_exe,
            )
        except Exception as exc:
            print(f'DIAMOND homology graph failed ({exc}); using k-mer fallback graph.')
            edges = []
    else:
        print('DIAMOND not found for homology split; using k-mer fallback graph.')

    if not edges:
        edges = _run_pairwise_kmer_edges(all_labelled_ids, seq_dict)

    component_map = _build_component_map_from_edges(all_labelled_ids, edges)
    group_labels = np.asarray([component_map[pid] for pid in all_labelled_ids], dtype=np.int64)

    if len(np.unique(group_labels)) < 2:
        print('Homology graph produced <2 components; using taxon-stratified fallback split.')
        train_ids, test_ids = build_taxon_stratified_split(
            all_labelled_ids,
            train_tax_map,
            test_size=test_size,
            seed=seed,
            cache_path=None,
        )
    else:
        from sklearn.model_selection import GroupShuffleSplit

        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        x_dummy = np.zeros((len(all_labelled_ids), 1), dtype=np.int8)
        train_idx, test_idx = next(gss.split(x_dummy, groups=group_labels))
        train_ids = sorted([all_labelled_ids[i] for i in train_idx])
        test_ids = sorted([all_labelled_ids[i] for i in test_idx])

    _save_split(cache_path, mode='homology_80_20', train_ids=train_ids, test_ids=test_ids)
    component_cache_path.write_text(json.dumps(component_map))
    print(
        f'Homology split ready: train={len(train_ids):,} test={len(test_ids):,} '
        f'components={len(set(component_map.values())):,}'
    )
    return train_ids, test_ids, component_map

In [ ]:
# All proteins that have both a sequence and at least one labelled GO term
all_labelled_ids_full = sorted(prot_to_terms.keys())
_train_tax_map_all = dict(zip(train_tax['protein'].values, train_tax['taxon'].values))

if USE_DEV_SUBSET:
    all_labelled_ids = sample_labelled_subset(
        all_labelled_ids_full,
        _train_tax_map_all,
        subset_size=DEV_SUBSET_TOTAL,
        seed=SEED,
        cache_path=DEV_SUBSET_CACHE_PATH,
    )
    print(f'Dev subset enabled: using {len(all_labelled_ids):,} labelled proteins out of {len(all_labelled_ids_full):,} total')
else:
    all_labelled_ids = all_labelled_ids_full
    print(f'Full dataset mode: using all {len(all_labelled_ids):,} labelled proteins')

split_registry = {}
split_registry['taxon_80_20'] = build_taxon_stratified_split(
    all_labelled_ids,
    _train_tax_map_all,
    test_size=TEST_SPLIT_RATIO,
    seed=SEED,
    cache_path=TAXON_SPLIT_CACHE_PATH,
)

split_registry['homology_80_20'] = build_homology_aware_split(
    all_labelled_ids,
    train_seqs,
    _train_tax_map_all,
    test_size=TEST_SPLIT_RATIO,
    seed=SEED,
    cache_path=HOMOLOGY_SPLIT_CACHE_PATH,
    work_dir=WORK_DIR,
    diamond_path=DIAMOND_PATH,
    evalue=HOMOLOGY_SPLIT_EVALUE,
)
homology_component_map = split_registry['homology_80_20'][2]

train_protein_ids, test_protein_ids = split_registry[SPLIT_MODE][:2]
test_tax_map = {pid: int(_train_tax_map_all.get(pid, -1)) for pid in test_protein_ids}

print(f'Active split mode: {SPLIT_MODE}')
print(f'Total labelled proteins in active pool:   {len(all_labelled_ids)}')
for _split_name in REPORT_SPLIT_MODES:
    _train_ids_rep, _test_ids_rep = split_registry[_split_name][:2]
    print(f'  {_split_name}: train={len(_train_ids_rep):,}  test={len(_test_ids_rep):,}  ratio={len(_test_ids_rep) / len(all_labelled_ids):.2%}')
print(f'Using -> train={len(train_protein_ids):,}  test={len(test_protein_ids):,}')


Dev subset enabled: using 500 labelled proteins out of 82,404 total
Active split mode: homology_80_20
Total labelled proteins in active pool:   500
  homology_80_20: train=400  test=100  ratio=20.00%
  taxon_80_20: train=400  test=100  ratio=20.00%
Using -> train=400  test=100


## GO DAG parsing + ancestor map (for score propagation)
Evaluation propagates child predictions to parents by max-score. We do the same, so the top-1500 pruning is more consistent with evaluation.

In [ ]:
print('Reading GO graph...')
go_graph = obonet.read_obo(str(go_obo))
print('Nodes:', go_graph.number_of_nodes(), 'Edges:', go_graph.number_of_edges())

# Precompute ancestors for all train terms (and optionally for IA terms)
all_terms_for_anc = set(train_terms['term'].unique().tolist())

anc_cache_path = WORK_DIR / 'ancestors_cache.json'

if anc_cache_path.exists():
    ancestors_map = {k: set(v) for k, v in json.loads(anc_cache_path.read_text()).items()}
    print('Loaded cached ancestors:', len(ancestors_map))
else:
    ancestors_map = {}
    missing_terms = 0
    for term in tqdm(sorted(all_terms_for_anc), desc='Ancestors'):
        if term not in go_graph:
            missing_terms += 1
            continue
        # obonet edges are typically from child -> parent for is_a/part_of relationships
        ancestors_map[term] = set(nx.ancestors(go_graph, term))
    anc_cache_path.write_text(json.dumps({k: sorted(list(v)) for k, v in ancestors_map.items()}))
    print('Missing in GO graph:', missing_terms)

def propagate_scores(term_to_score: dict, ancestors: dict) -> dict:
    out = dict(term_to_score)
    for term, score in term_to_score.items():
        for parent in ancestors.get(term, ()): 
            prev = out.get(parent)
            if (prev is None) or (score > prev):
                out[parent] = score
    return out

Reading GO graph...
Nodes: 40122 Edges: 77229


Ancestors:   0%|          | 0/26125 [00:00<?, ?it/s]

Missing in GO graph: 0


## ESM2 embeddings
We compute one fixed-length embedding per protein using either last-layer mean pooling or concatenated mean pools from the last 3 hidden layers, depending on `POOLING_MODE`.

Long sequences are chunked into pieces of length `MAX_AA_PER_CHUNK` and averaged.

In [ ]:
print(f'Embedding backbone: {MODEL_NAME} | pooling={POOLING_MODE}')
print(f'Max AA per chunk: {MAX_AA_PER_CHUNK} | batch size: {BATCH_SIZE}')


Embedding backbone: facebook/esm2_t33_650M_UR50D | pooling=last3_concat
Max AA per chunk: 1000 | batch size: 4


In [ ]:
# Embedding & backbone helper functions
def _clean_protein_sequence(seq: str) -> str:
    """Replace non-standard amino acids with X."""
    import re
    return re.sub(r'[UZOB]', 'X', seq)

def _prep_prott5(seq: str) -> str:
    """Add spaces between amino acids for ProtT5 tokenizer."""
    return ' '.join(list(_clean_protein_sequence(seq)))

@torch.no_grad()
def embed_batch(
    tokenizer,
    model,
    seqs: list[str],
    *,
    device: str,
    is_prott5: bool = False,
    pooling_mode: str = "last_mean",
) -> np.ndarray:
    """Compute embeddings for a batch of sequences."""
    if not is_prott5:
        seqs = [_clean_protein_sequence(seq) for seq in seqs]
    if is_prott5:
        seqs = [_prep_prott5(seq) for seq in seqs]

    toks = tokenizer(seqs, return_tensors="pt", padding=True, truncation=True)
    toks = {k: v.to(device) for k, v in toks.items()}
    use_hidden_states = pooling_mode == "last3_concat"
    
    # T5 models are encoder-decoder; for embedding extraction use encoder only
    if is_prott5 and hasattr(model, 'encoder'):
        out = model.encoder(**toks, output_hidden_states=use_hidden_states)
    else:
        out = model(**toks, output_hidden_states=use_hidden_states)
    
    attn = toks.get("attention_mask")

    if pooling_mode == "last3_concat":
        # Average the last 3 hidden states
        hidden_list = [out.hidden_states[-i] for i in [3, 2, 1]]  # Last 3 layers
        pooled = [_masked_mean(hidden, attn) for hidden in hidden_list]
        emb = torch.cat(pooled, dim=1)
    else:
        emb = _masked_mean(out.last_hidden_state, attn)

    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb.detach().cpu().numpy()

def _masked_mean(hidden: torch.Tensor, attn: torch.Tensor | None) -> torch.Tensor:
    """Compute masked mean of hidden states."""
    if hidden.size(1) > 2:
        hidden_use = hidden[:, 1:-1, :]
        attn_use = attn[:, 1:-1] if attn is not None else None
    else:
        hidden_use = hidden
        attn_use = attn

    if attn_use is None:
        return hidden_use.mean(dim=1)

    mask = attn_use.unsqueeze(-1).to(hidden_use.dtype)
    summed = (hidden_use * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return summed / denom

def chunk_sequence(seq: str, max_len: int) -> list[str]:
    """Split a sequence into chunks."""
    if len(seq) <= max_len:
        return [seq]
    return [seq[i : i + max_len] for i in range(0, len(seq), max_len)]

def compute_embeddings(
    protein_ids: list[str],
    seq_dict: dict[str, str],
    *,
    tokenizer,
    model,
    emb_path,
    ids_path,
    device: str,
    batch_size: int,
    max_aa_per_chunk: int,
    pooling_mode: str = "last_mean",
    is_prott5: bool = False,
) -> tuple[list[str], np.ndarray]:
    """Compute or load cached embeddings."""
    protein_ids = list(protein_ids)
    if emb_path.exists() and ids_path.exists():
        cached_ids = json.loads(ids_path.read_text())
        emb = np.load(emb_path)
        if cached_ids == protein_ids and len(cached_ids) == emb.shape[0]:
            print("Loaded cached embeddings:", emb.shape)
            return cached_ids, emb

    ids_out: list[str] = []
    emb_out: list[np.ndarray] = []
    simple_ids: list[str] = []
    simple_seqs: list[str] = []

    def flush_simple() -> None:
        nonlocal simple_ids, simple_seqs
        if not simple_ids:
            return
        for start in range(0, len(simple_seqs), batch_size):
            batch_ids = simple_ids[start : start + batch_size]
            batch_seqs = simple_seqs[start : start + batch_size]
            batch_emb = embed_batch(
                tokenizer,
                model,
                batch_seqs,
                device=device,
                is_prott5=is_prott5,
                pooling_mode=pooling_mode,
            )
            ids_out.extend(batch_ids)
            emb_out.append(batch_emb)
        simple_ids, simple_seqs = [], []

    for pid in tqdm(protein_ids, desc=f"Embedding ({emb_path.name})"):
        seq = seq_dict[pid]
        chunks = chunk_sequence(seq, max_aa_per_chunk)
        if len(chunks) == 1:
            simple_ids.append(pid)
            simple_seqs.append(chunks[0])
            continue

        flush_simple()
        chunk_embs = []
        chunk_lengths = []
        for start in range(0, len(chunks), batch_size):
            batch_chunks = chunks[start : start + batch_size]
            chunk_embs.append(
                embed_batch(
                    tokenizer,
                    model,
                    batch_chunks,
                    device=device,
                    is_prott5=is_prott5,
                    pooling_mode=pooling_mode,
                )
            )
            chunk_lengths.extend([len(c) for c in batch_chunks])

        prot_emb = np.average(np.vstack(chunk_embs), axis=0, weights=chunk_lengths)
        ids_out.append(pid)
        emb_out.append(prot_emb[np.newaxis, :])

    flush_simple()
    emb = np.vstack(emb_out)
    json.dumps(ids_out)  # Validate JSON-able
    ids_path.write_text(json.dumps(ids_out))
    np.save(emb_path, emb)
    print(f"Saved embeddings: {emb.shape}")
    return ids_out, emb

In [ ]:
# Protein ID lists come from the 80:20 split defined above
train_ids = train_protein_ids   # 80 % — used for kNN index + supervised head
test_ids  = test_protein_ids    # 20 % — used only for evaluation (has true labels)

test_seqs = {pid: train_seqs[pid] for pid in test_ids}

esm2_tokenizer, esm2_model = load_hf_backbone(
    MODEL_NAME,
    device=DEVICE,
    output_hidden_states=(POOLING_MODE == 'last3_concat'),
)

train_ids_emb, train_emb = compute_embeddings(
    train_ids,
    train_seqs,
    tokenizer=esm2_tokenizer,
    model=esm2_model,
    emb_path=TRAIN_EMB_PATH,
    ids_path=TRAIN_IDS_PATH,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_aa_per_chunk=MAX_AA_PER_CHUNK,
    pooling_mode=POOLING_MODE,
    is_prott5=False,
)

# Test proteins come from train_seqs (they are the held-out 20 %)
test_ids_emb, test_emb = compute_embeddings(
    test_ids,
    train_seqs,
    tokenizer=esm2_tokenizer,
    model=esm2_model,
    emb_path=TEST_SPLIT_EMB_PATH,
    ids_path=TEST_SPLIT_IDS_PATH,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    max_aa_per_chunk=MAX_AA_PER_CHUNK,
    pooling_mode=POOLING_MODE,
    is_prott5=False,
)

assert train_ids_emb == train_ids, 'Train embedding order mismatch'
assert test_ids_emb  == test_ids,  'Test  embedding order mismatch'
print('ESM2 train emb:', train_emb.shape, ' | ESM2 test-split emb:', test_emb.shape)

# Optional second backbone (ProtT5)
prott5_train_emb = None
prott5_test_emb  = None
if USE_PROTT5:
    if DEVICE != 'cuda':
        raise RuntimeError('ProtT5 is very large; please run with a GPU (DEVICE=cuda).')

    prott5_tokenizer, prott5_model = load_hf_backbone(
        PROTT5_MODEL_NAME,
        device=DEVICE,
        output_hidden_states=(POOLING_MODE == 'last3_concat'),
    )

    prott5_train_ids_emb, prott5_train_emb = compute_embeddings(
        train_ids,
        train_seqs,
        tokenizer=prott5_tokenizer,
        model=prott5_model,
        emb_path=PROTT5_TRAIN_EMB_PATH,
        ids_path=PROTT5_TRAIN_IDS_PATH,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_aa_per_chunk=MAX_AA_PER_CHUNK,
        pooling_mode=POOLING_MODE,
        is_prott5=True,
    )
    prott5_test_ids_emb, prott5_test_emb = compute_embeddings(
        test_ids,
        train_seqs,
        tokenizer=prott5_tokenizer,
        model=prott5_model,
        emb_path=PROTT5_TEST_SPLIT_EMB_PATH,
        ids_path=PROTT5_TEST_SPLIT_IDS_PATH,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_aa_per_chunk=MAX_AA_PER_CHUNK,
        pooling_mode=POOLING_MODE,
        is_prott5=True,
    )

    assert prott5_train_ids_emb == train_ids
    assert prott5_test_ids_emb  == test_ids
    print('ProtT5 train emb:', prott5_train_emb.shape, ' | ProtT5 test-split emb:', prott5_test_emb.shape)


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

2026-04-06 00:15:22.229528: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775434522.408544      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775434522.461062      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775434522.899272      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775434522.899322      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775434522.899325      55 computation_placer.cc:177] computation placer alr

...

Embedding (train_emb_Rostlab_prot_t5_xl_uniref50_homology_80_20_dev500_seed472_split20.npy):   0%|          | …

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Saved embeddings: (400, 3072)


Embedding (test_split_emb_Rostlab_prot_t5_xl_uniref50_homology_80_20_dev500_seed472_split20.npy):   0%|       …

Saved embeddings: (100, 3072)
ProtT5 train emb: (400, 3072)  | ProtT5 test-split emb: (100, 3072)


## Step — Sequence-similarity signal (DIAMOND / k-mer fallback)

Builds a dictionary `seq_sim_hits = {test_pid: [(train_pid, norm_score), ...]}` using:

1. **DIAMOND blastp** (fastest, highest quality) if `diamond` is on PATH or `DIAMOND_PATH` is set
2. **NCBI blastp** as a secondary option
3. **k-mer cosine similarity** (TF-IDF over amino-acid 3-grams) as a pure-Python fallback — no external dependencies required

Reference: Radivojac et al., *Nature Methods* (2013) — BLAST-based homology transfer remains a top baseline in CAFA challenges, dominating BPO.


In [ ]:
import subprocess, shutil, tempfile, pickle
from collections import defaultdict
from pathlib import Path

# -----------------------------------------------------------------------
# helpers: FASTA write & parse DIAMOND/BLAST tabular output
# -----------------------------------------------------------------------
def write_fasta_legacy(ids: list, seq_dict: dict, path: Path) -> None:
    with open(path, 'w') as fh:
        for pid in ids:
            seq = seq_dict.get(pid, '')
            if seq:
                fh.write(f'>{pid}\n{seq}\n')


def parse_tabular_hits_legacy(tsv_path: Path, top_k: int, evalue_col: int = None) -> dict:
    """
    Parse DIAMOND/BLAST -outfmt 6 output (qseqid sseqid bitscore).
    Returns {qid: [(sid, norm_score), ...]} with scores in (0, 1].
    """
    raw = defaultdict(list)
    with open(tsv_path) as fh:
        for line in fh:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            qid, sid, bs = parts[0], parts[1], parts[2]
            try:
                bs = float(bs)
            except ValueError:
                continue
            if qid != sid:           # skip self-hits
                raw[qid].append((sid, bs))

    hits = {}
    for qid, lst in raw.items():
        lst.sort(key=lambda x: -x[1])
        lst = lst[:top_k]
        if not lst:
            continue
        max_bs = lst[0][1] + 1e-12
        hits[qid] = [(sid, bs / max_bs) for sid, bs in lst]
    return hits


# -----------------------------------------------------------------------
# Method 1 — DIAMOND blastp
# -----------------------------------------------------------------------
def run_diamond_legacy(train_ids, test_ids, train_seqs, test_seqs,
                top_k, evalue, diamond_exe) -> dict | None:
    try:
        tmp = Path(tempfile.mkdtemp())
        db_fasta = tmp / 'train.fasta'
        q_fasta  = tmp / 'query.fasta'
        db_path  = tmp / 'traindb'
        out_tsv  = tmp / 'hits.tsv'

        write_fasta(train_ids, train_seqs, db_fasta)
        write_fasta(test_ids,  test_seqs,  q_fasta)

        # make database
        subprocess.run(
            [diamond_exe, 'makedb', '--in', str(db_fasta),
             '--db', str(db_path), '--quiet'],
            check=True, capture_output=True, timeout=600
        )
        # run blastp
        subprocess.run(
            [diamond_exe, 'blastp',
             '--query', str(q_fasta),
             '--db', str(db_path),
             '--out', str(out_tsv),
             '--outfmt', '6', 'qseqid', 'sseqid', 'bitscore',
             '--max-target-seqs', str(top_k),
             '--evalue', str(evalue),
             '--threads', '4', '--quiet'],
            check=True, capture_output=True, timeout=7200
        )
        hits = parse_tabular_hits(out_tsv, top_k)
        print(f'  DIAMOND: {sum(len(v) for v in hits.values())} hits for '
              f'{len(hits)}/{len(test_ids)} query proteins')
        return hits
    except Exception as e:
        print(f'  DIAMOND failed ({e}); trying next method...')
        return None
    finally:
        import shutil as _sh
        _sh.rmtree(tmp, ignore_errors=True)


# -----------------------------------------------------------------------
# Method 2 — NCBI blastp
# -----------------------------------------------------------------------
def run_blastp_legacy(train_ids, test_ids, train_seqs, test_seqs,
               top_k, evalue) -> dict | None:
    if shutil.which('blastp') is None and shutil.which('makeblastdb') is None:
        print('  blastp not found; skipping...')
        return None
    try:
        tmp = Path(tempfile.mkdtemp())
        db_fasta = tmp / 'train.fasta'
        q_fasta  = tmp / 'query.fasta'
        db_path  = tmp / 'traindb'
        out_tsv  = tmp / 'hits.tsv'

        write_fasta(train_ids, train_seqs, db_fasta)
        write_fasta(test_ids,  test_seqs,  q_fasta)

        subprocess.run(
            ['makeblastdb', '-in', str(db_fasta), '-dbtype', 'prot',
             '-out', str(db_path)],
            check=True, capture_output=True, timeout=600
        )
        subprocess.run(
            ['blastp',
             '-query', str(q_fasta),
             '-db', str(db_path),
             '-outfmt', '6 qseqid sseqid bitscore',
             '-max_target_seqs', str(top_k),
             '-evalue', str(evalue),
             '-num_threads', '4',
             '-out', str(out_tsv)],
            check=True, capture_output=True, timeout=7200
        )
        hits = parse_tabular_hits(out_tsv, top_k)
        print(f'  BLAST: {sum(len(v) for v in hits.values())} hits for '
              f'{len(hits)}/{len(test_ids)} query proteins')
        return hits
    except Exception as e:
        print(f'  BLASTP failed ({e}); falling back to k-mer...')
        return None
    finally:
        import shutil as _sh
        _sh.rmtree(tmp, ignore_errors=True)


# -----------------------------------------------------------------------
# Method 3 — k-mer cosine similarity (pure Python fallback)
# -----------------------------------------------------------------------
def run_kmer_similarity_legacy(train_ids, test_ids, train_seqs, test_seqs,
                        top_k) -> dict:
    """
    Builds a TF-IDF matrix over amino-acid character 3-grams for all
    proteins, then computes cosine similarity between each test and all
    train proteins.  Returns {test_pid: [(train_pid, sim), ...]}.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    import scipy.sparse as sp

    print(f'  Building k-mer matrix for {len(train_ids)} train + '
          f'{len(test_ids)} test proteins ...')

    all_ids  = list(train_ids) + list(test_ids)
    all_seqs = []
    for pid in all_ids:
        seq = train_seqs.get(pid) or test_seqs.get(pid, 'X')
        all_seqs.append(seq)

    vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 3),
                          min_df=2, sublinear_tf=True)
    mat = vec.fit_transform(all_seqs)

    n_train = len(train_ids)
    train_mat = normalize(mat[:n_train], norm='l2')
    test_mat  = normalize(mat[n_train:], norm='l2')

    hits  = {}
    batch = 256
    t_ids = list(train_ids)

    for i in range(0, len(test_ids), batch):
        batch_ids = test_ids[i:i + batch]
        sims      = (test_mat[i:i + batch] @ train_mat.T).toarray()

        for row, pid in enumerate(batch_ids):
            sim_row = sims[row]
            top_idx = np.argsort(-sim_row)[:top_k]
            top_hits = [(t_ids[j], float(sim_row[j]))
                        for j in top_idx if sim_row[j] > 0]
            if top_hits:
                hits[pid] = top_hits

    print(f'  k-mer: {sum(len(v) for v in hits.values())} hits for '
          f'{len(hits)}/{len(test_ids)} query proteins')
    return hits


# -----------------------------------------------------------------------
# Main dispatcher — try DIAMOND → BLAST → k-mer
# -----------------------------------------------------------------------
def compute_seq_sim_hits_legacy(train_ids, test_ids, train_seqs, test_seqs,
                         top_k, evalue, diamond_path='') -> dict:
    # 1. locate diamond executable
    diamond_exe = None
    if diamond_path:
        diamond_exe = diamond_path
    else:
        for candidate in ['diamond', 'diamond.exe']:
            if shutil.which(candidate):
                diamond_exe = candidate
                break

    if diamond_exe:
        print(f'  Found DIAMOND ({diamond_exe})')
        result = run_diamond(train_ids, test_ids, train_seqs, test_seqs,
                             top_k, evalue, diamond_exe)
        if result is not None:
            return result

    # 2. try NCBI blastp
    result = run_blastp(train_ids, test_ids, train_seqs, test_seqs,
                        top_k, evalue)
    if result is not None:
        return result

    # 3. k-mer fallback
    print('  Using k-mer cosine similarity (no DIAMOND/BLAST found)')
    return run_kmer_similarity(train_ids, test_ids, train_seqs, test_seqs,
                               top_k)


# -----------------------------------------------------------------------
# Compute / load from cache
# -----------------------------------------------------------------------
seq_sim_hits: dict = {}

if USE_SEQ_SIM:
    if SEQ_SIM_CACHE_PATH.exists():
        print(f'Loading seq-sim hits from cache: {SEQ_SIM_CACHE_PATH.name}')
        with open(SEQ_SIM_CACHE_PATH, 'rb') as fh:
            seq_sim_hits = pickle.load(fh)
        print(f'  Loaded {len(seq_sim_hits)} entries (top-{SEQ_SIM_TOP_K} hits each)')
    else:
        print('Computing sequence-similarity hits...')
        seq_sim_hits = compute_seq_sim_hits(
            train_ids   = train_ids,
            test_ids    = test_ids,       # validation split (or final test)
            train_seqs  = train_seqs,
            test_seqs   = test_seqs,
            top_k       = SEQ_SIM_TOP_K,
            evalue      = SEQ_SIM_EVALUE,
            diamond_path= DIAMOND_PATH,
        )
        with open(SEQ_SIM_CACHE_PATH, 'wb') as fh:
            pickle.dump(seq_sim_hits, fh, protocol=4)
        print(f'  Cached to {SEQ_SIM_CACHE_PATH.name}')

    if seq_sim_hits:
        n_covered = len(seq_sim_hits)
        n_total   = len(test_ids)
        avg_hits  = sum(len(v) for v in seq_sim_hits.values()) / max(n_covered, 1)
        print(f'Seq-sim coverage: {n_covered}/{n_total} '
              f'({100*n_covered/n_total:.1f}%)  avg hits per protein: {avg_hits:.1f}')
else:
    print('USE_SEQ_SIM=False — skipping sequence-similarity signal.')


Computing sequence-similarity hits...
  Found DIAMOND (diamond)
  DIAMOND: 17 hits for 12/100 query proteins
  Cached to seq_sim_hits_homology_80_20_dev500_seed472_split20_top10_e0p001.pkl
Seq-sim coverage: 12/100 (12.0%)  avg hits per protein: 1.4


## Build a kNN index
We use cosine similarity on L2-normalized embeddings. If `faiss` is available, it will be used for speed.

In [ ]:
_faiss_available = None

def build_faiss_index(emb: np.ndarray):
    global _faiss_available
    if _faiss_available is None:
        try:
            import faiss as _faiss_mod
            _faiss_available = True
        except ImportError:
            _faiss_available = False
            print('FAISS not available — using brute-force dot-product kNN.')
    if _faiss_available:
        import faiss
        index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb.astype(np.float32))
        return index
    return None


def knn_search(query_emb: np.ndarray, train_emb: np.ndarray, index, k: int):
    """Returns (indices, similarities) — cosine if inputs are L2-normalised."""
    if index is not None:
        sims, idx = index.search(query_emb.astype(np.float32), k)
        return idx, sims
    sims = query_emb @ train_emb.T
    idx  = np.argsort(-sims, axis=1)[:, :k]
    return idx, np.take_along_axis(sims, idx, axis=1)


def _label_transfer_precomputed(
    idx: np.ndarray,
    sims: np.ndarray,
    term_lists: list,
    top_terms: int,
) -> list[dict]:
    """
    Inner kNN label-transfer loop operating on precomputed (idx, sims) arrays.
    Factored out from knn_label_transfer so that predict_knn_raw_scores_with_taxon
    can reuse a single knn_search result for both label transfer AND
    mean-similarity extraction (avoids double-searching for dynamic alpha).
    """
    all_preds = []
    for row in range(idx.shape[0]):
        neigh_idx  = idx[row]
        neigh_sims = np.maximum(sims[row], 0.0)

        scores = defaultdict(float)
        denom  = 0.0

        for rank, (j, si) in enumerate(zip(neigh_idx, neigh_sims)):
            if USE_RANK_WEIGHTED_KNN:
                w = 1.0 / (rank + 1)
            else:
                w = float(si)
                if w <= 0:
                    continue
            denom += w
            for term in term_lists[int(j)]:
                ia_w = sqrt_ia_map.get(term, 1.0) if USE_IA_WEIGHTED_KNN else 1.0
                scores[term] += w * ia_w

        if denom > 0:
            for t in list(scores.keys()):
                scores[t] /= denom

        if len(scores) > top_terms:
            scores = dict(sorted(scores.items(),
                                 key=lambda x: x[1], reverse=True)[:top_terms])
        else:
            scores = dict(scores)

        all_preds.append(scores)
    return all_preds


def knn_label_transfer(
    query_emb: np.ndarray,
    train_emb: np.ndarray,
    train_term_lists,
    index,
    k: int,
    top_terms_pre_prop: int,
):
    """Returns list[dict(term->score)] for each query row.

    Rank-weighted kNN (USE_RANK_WEIGHTED_KNN):
        Weight of neighbour at rank r  =  1 / (r + 1)

    IA-weighted term contribution (USE_IA_WEIGHTED_KNN):
        Each term's per-neighbour contribution is multiplied by
        sqrt_ia_map[term], biasing toward high-information leaf terms.
    """
    idx, sims = knn_search(query_emb, train_emb, index, k)
    return _label_transfer_precomputed(idx, sims, train_term_lists, top_terms_pre_prop)


## kNN label transfer (train → test split)
For a query protein, we take its top-$K$ nearest training proteins and aggregate their GO terms with similarity weights.

Scoring rule (simple + effective):
$$ score(\text{term}) = \frac{\sum_{i\in\mathrm{kNN}} \mathrm{sim}_i \cdot \mathbf{1}[\text{term}\in\text{labels}_i]}{\sum_{i\in\mathrm{kNN}} \mathrm{sim}_i + \epsilon} $$

Then we:
1. Keep top `TOP_TERMS_PER_PROTEIN_BEFORE_PROP` terms
2. Propagate to parents via GO DAG (max child score)
3. Keep top `MAX_TERMS_PER_PROTEIN_FINAL` terms
4. Evaluate predictions against true labels on the held-out 20 % test set


In [ ]:
# Build global kNN index on the full train set
train_term_lists = [prot_to_terms.get(pid, []) for pid in train_ids]
global_index = build_faiss_index(train_emb)


def blend_score_dicts(a: dict, b: dict, alpha: float) -> dict:
    # alpha * a + (1-alpha) * b
    out = dict(b)
    for k, v in a.items():
        out[k] = alpha * v + (1.0 - alpha) * out.get(k, 0.0)
    return out


def postprocess_scores(scores: dict) -> dict:
    """Prune → propagate to GO ancestors → prune → filter to IA terms → max-normalise."""
    # 1. Keep top-N before propagation (reduces ancestor explosion)
    if len(scores) > TOP_TERMS_PER_PROTEIN_BEFORE_PROP:
        scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True)
                      [:TOP_TERMS_PER_PROTEIN_BEFORE_PROP])

    # 2. Propagate child scores to all GO ancestors (max-child rule)
    scores = propagate_scores(scores, ancestors_map)

    # 3. Keep final top-N
    if len(scores) > MAX_TERMS_PER_PROTEIN_FINAL:
        scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True)
                      [:MAX_TERMS_PER_PROTEIN_FINAL])

    # 4. Remove terms without IA weights — they count against precision but
    #    contribute 0 to recall, so keeping them only hurts the score.
    scores = {t: s for t, s in scores.items() if t in ia_terms}

    # 5. Per-protein max-normalisation: highest-confidence term → score 1.0.
    #    This makes the threshold sweep consistent across proteins and helps
    #    the supervised-head and kNN contributions use the same scale.
    if USE_PER_PROTEIN_MAX_NORMALIZATION and scores:
        max_s = max(scores.values())
        if max_s > 0:
            scores = {t: s / max_s for t, s in scores.items()}

    return scores


def predict_knn_scores(query_emb: np.ndarray, *, k: int) -> list[dict]:
    preds = knn_label_transfer(
        query_emb=query_emb,
        train_emb=train_emb,
        train_term_lists=train_term_lists,
        index=global_index,
        k=k,
        top_terms_pre_prop=TOP_TERMS_PER_PROTEIN_BEFORE_PROP,
    )
    return [postprocess_scores(d) for d in preds]


FAISS not available — using brute-force dot-product kNN.


## (Optional) Validation: compute IA-weighted max-F on a group split
This is for model selection / debugging. Kaggle evaluation uses max-F across thresholds anyway, but validating helps tune hyperparameters (K, pooling, propagation/pruning).

In [ ]:
ia_map = dict(zip(ia['term'].values, ia['ia'].values))

def weighted_prf(true_terms: set, pred_terms: set):
    # weighted precision/recall using IA weights; true/pred are sets of GO terms
    if len(pred_terms) == 0:
        return 0.0, 0.0, 0.0
    inter = true_terms & pred_terms
    wp_num = sum(ia_map.get(t, 0.0) for t in inter)
    wp_den = sum(ia_map.get(t, 0.0) for t in pred_terms) + 1e-12
    wr_den = sum(ia_map.get(t, 0.0) for t in true_terms) + 1e-12
    wp = wp_num / wp_den
    wr = wp_num / wr_den
    if wp + wr == 0:
        wf = 0.0
    else:
        wf = 2 * wp * wr / (wp + wr)
    return wp, wr, wf

def max_f_over_thresholds(true_by_prot, pred_scores_by_prot, ontology_filter=None, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 50)
    best = (0.0, None)
    for th in thresholds:
        fs = []
        for pid in true_by_prot.keys():
            true_terms = true_by_prot[pid]
            if ontology_filter is not None:
                true_terms = {t for t in true_terms if term_to_ont.get(t) == ontology_filter}
            pred_terms = {t for t, s in pred_scores_by_prot[pid].items() if s >= th}
            if ontology_filter is not None:
                pred_terms = {t for t in pred_terms if term_to_ont.get(t) == ontology_filter}
            _, _, f = weighted_prf(true_terms, pred_terms)
            fs.append(f)
        mf = float(np.mean(fs))
        if mf > best[0]:
            best = (mf, th)
    return best

# Build an inner fit/validation split. Use homology groups when available,
# otherwise fall back to taxon-level grouping.
train_tax_map = dict(zip(train_tax['protein'].values, train_tax['taxon'].values))
if SPLIT_MODE == 'homology_80_20':
    homology_groups = np.array([homology_component_map.get(pid, -1) for pid in train_ids])
    if len(np.unique(homology_groups)) > 1:
        train_groups = homology_groups
    else:
        print('Homology groups unavailable in cache; falling back to taxon groups for inner split.')
        train_groups = np.array([train_tax_map.get(pid, -1) for pid in train_ids])
else:
    train_groups = np.array([train_tax_map.get(pid, -1) for pid in train_ids])

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
fit_idx, val_idx = next(gss.split(train_emb, groups=train_groups))
fit_ids = [train_ids[i] for i in fit_idx]
val_ids = [train_ids[i] for i in val_idx]

print('Fit:', len(fit_ids), 'Val:', len(val_ids))

Homology groups unavailable in cache; falling back to taxon groups for inner split.
Fit: 378 Val: 22


## Step 1 — Increase `K_NEIGHBORS` and tune it via validation
We tune $K$ on a **group split by taxon** (reduces leakage). This runs only if `TUNE_K=True`.

In [ ]:
# Grid-search K using the fit/val split computed above.
# We evaluate mean IA-weighted max-F across ontologies.

if TUNE_K:
    max_k = max(K_GRID)

    fit_emb = train_emb[fit_idx]
    val_emb = train_emb[val_idx]
    fit_term_lists = [prot_to_terms.get(pid, []) for pid in fit_ids]
    fit_index = build_faiss_index(fit_emb)

    # Precompute neighbors once for max_k, then reuse prefixes for smaller K
    val_nn_idx, val_nn_sims = knn_search(val_emb, fit_emb, fit_index, max_k)

    true_val = {pid: set(prot_to_terms.get(pid, [])) for pid in val_ids}

    def preds_for_k(k: int):
        preds = {}
        for r, pid in enumerate(val_ids):
            neigh_idx = val_nn_idx[r, :k]
            neigh_sims = val_nn_sims[r, :k]
            neigh_sims = np.maximum(neigh_sims, 0.0)
            denom = float(neigh_sims.sum()) + 1e-12

            scores = defaultdict(float)
            for j, si in zip(neigh_idx, neigh_sims):
                if si <= 0:
                    continue
                for term in fit_term_lists[int(j)]:
                    scores[term] += float(si)
            for t in list(scores.keys()):
                scores[t] /= denom

            if len(scores) > TOP_TERMS_PER_PROTEIN_BEFORE_PROP:
                scores = dict(sorted(scores.items(), key=lambda x: x[1], reverse=True)[:TOP_TERMS_PER_PROTEIN_BEFORE_PROP])
            else:
                scores = dict(scores)

            preds[pid] = postprocess_scores(scores)
        return preds

    results = []
    for k in K_GRID:
        preds_val = preds_for_k(k)
        per_ont = {}
        for o in ['MFO', 'BPO', 'CCO']:
            best_f, best_th = max_f_over_thresholds(true_val, preds_val, ontology_filter=o)
            per_ont[o] = best_f
        mean_f = float(np.mean(list(per_ont.values())))
        results.append((mean_f, k, per_ont))
        print(f'K={k}: mean max-F={mean_f:.4f} |', per_ont)

    results.sort(reverse=True, key=lambda x: x[0])
    best_mean_f, best_k, best_breakdown = results[0]
    print('Best K:', best_k, 'mean max-F:', best_mean_f, 'breakdown:', best_breakdown)

    K_NEIGHBORS = best_k
    print('Updated K_NEIGHBORS =', K_NEIGHBORS)


## Step 2 — Taxonomy-aware ensemble (within-taxon kNN + global kNN)
We build a separate kNN index per taxon (when enough training proteins exist), then blend taxon-local scores with global scores.

In [ ]:
# Build taxon-specific kNN indices (ESM2 backbone)
train_tax_map = dict(zip(train_tax['protein'].values, train_tax['taxon'].values))

# taxon -> list of train row indices
_taxon_to_rows = defaultdict(list)
for i, pid in enumerate(train_ids):
    tx = train_tax_map.get(pid, -1)
    _taxon_to_rows[int(tx)].append(i)

# Build indices only for taxa with enough proteins
# Store: taxon -> (train_emb_sub, train_term_lists_sub, index)
taxon_knn = {}
if USE_TAXON_ENSEMBLE:
    for tx, rows in tqdm(list(_taxon_to_rows.items()), desc='Build taxon indices'):
        if tx == -1:
            continue
        if len(rows) < MIN_TAXON_TRAIN:
            continue
        emb_sub = train_emb[np.array(rows)]
        term_lists_sub = [train_term_lists[i] for i in rows]
        idx_sub = build_faiss_index(emb_sub)
        taxon_knn[tx] = (emb_sub, term_lists_sub, idx_sub)

    print('Built taxon-specific indices:', len(taxon_knn))

# -----------------------------------------------------------------------
# Option 1 — Bayesian smoothing: precompute α_bayes per taxon
#   α_bayes(t) = N_t / (N_t + λ)
#   λ = TAXON_BAYESIAN_LAMBDA  (default 100)
#   A taxon with 100 training proteins gets α_bayes = 0.50
#   A taxon with 500 proteins gets α_bayes = 0.83
#   A taxon with 5000 proteins gets α_bayes = 0.98
# -----------------------------------------------------------------------
taxon_bayesian_alpha = {}
if USE_TAXON_ENSEMBLE and USE_DYNAMIC_TAXON_ALPHA:
    for tx in taxon_knn:
        n_t = len(_taxon_to_rows.get(tx, []))
        taxon_bayesian_alpha[tx] = n_t / (n_t + TAXON_BAYESIAN_LAMBDA)
    if taxon_bayesian_alpha:
        alphas = list(taxon_bayesian_alpha.values())
        print(f'Bayesian alpha — range: {min(alphas):.3f} – {max(alphas):.3f}  '
              f'median: {float(np.median(alphas)):.3f}  '
              f'(λ={TAXON_BAYESIAN_LAMBDA}, {len(taxon_bayesian_alpha)} taxa)')


def predict_knn_raw_scores_with_taxon(ids: list[str], emb: np.ndarray, *, k: int) -> list[dict]:
    # -------------------------------------------------------------------
    # 1) Global search — one knn_search call reused for BOTH label
    #    transfer and mean-similarity (for Option 2 per-query alpha).
    # -------------------------------------------------------------------
    global_idx, global_sims = knn_search(emb, train_emb, global_index, k)
    global_raw = _label_transfer_precomputed(
        global_idx, global_sims, train_term_lists, TOP_TERMS_PER_PROTEIN_BEFORE_PROP
    )
    # Per-query mean top-K cosine similarity — Option 2 signal
    global_mean_sim = np.maximum(global_sims, 0.0).mean(axis=1)  # [n_query]

    if not USE_TAXON_ENSEMBLE:
        return global_raw

    # -------------------------------------------------------------------
    # 2) Taxon-specific blend with dynamic alpha
    # -------------------------------------------------------------------
    blended = list(global_raw)

    by_taxon = defaultdict(list)
    for pos, pid in enumerate(ids):
        tx = int(test_tax_map.get(pid, -1))
        by_taxon[tx].append(pos)

    for tx, positions in by_taxon.items():
        struct = taxon_knn.get(tx)
        if struct is None:
            continue
        emb_sub, term_lists_sub, idx_sub = struct

        q_emb = emb[np.array(positions)]
        tk = min(k, emb_sub.shape[0])

        # Single taxon search reused for label transfer + mean sim
        taxon_idx, taxon_sims = knn_search(q_emb, emb_sub, idx_sub, tk)
        taxon_mean_sim = np.maximum(taxon_sims, 0.0).mean(axis=1)  # [len(positions)]
        tax_raw = _label_transfer_precomputed(
            taxon_idx, taxon_sims, term_lists_sub, TOP_TERMS_PER_PROTEIN_BEFORE_PROP
        )

        for local_i, pos in enumerate(positions):
            if USE_DYNAMIC_TAXON_ALPHA:
                # Option 2 — per-query similarity ratio
                s_t = float(taxon_mean_sim[local_i])
                s_g = float(global_mean_sim[pos])
                alpha_sim = s_t / (s_t + s_g + 1e-12)

                # Option 1 — Bayesian population floor
                alpha_bayes = taxon_bayesian_alpha.get(tx, TAXON_BLEND_ALPHA)

                # Combined: per-query retrieve confidence × population trust
                # Effect: high-similarity query in large taxon → α near 1.0
                #         high-similarity query in tiny taxon → α scaled down
                #         low-similarity query regardless    → α pulled toward 0.5
                alpha = alpha_sim * alpha_bayes
            else:
                alpha = TAXON_BLEND_ALPHA

            blended[pos] = blend_score_dicts(tax_raw[local_i], blended[pos], alpha)

    return blended


def predict_knn_scores_final(ids: list[str], emb: np.ndarray, *, k: int) -> list[dict]:
    raw = predict_knn_raw_scores_with_taxon(ids, emb, k=k)
    return [postprocess_scores(d) for d in raw]


Build taxon indices:   0%|          | 0/50 [00:00<?, ?it/s]

Built taxon-specific indices: 0


## Step 3 — Add a lightweight supervised head (per ontology)
We train a simple **linear classifier** on top of embeddings for the most frequent GO terms per ontology, then blend its probabilities with kNN scores.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer


def _sigmoid_legacy(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


def build_supervised_terms_legacy(train_terms_df: pd.DataFrame, n_per_ont):
    """n_per_ont can be int (same for all) or dict {ont: int}."""
    out = {}
    for o in ['MFO', 'BPO', 'CCO']:
        n = n_per_ont[o] if isinstance(n_per_ont, dict) else int(n_per_ont)
        df = train_terms_df[train_terms_df['ontology'] == o]
        out[o] = df['term'].value_counts().head(n).index.tolist()
    return out


def fit_or_load_supervised_head_legacy():
    if not USE_SUPERVISED_HEAD:
        return None

    if SUP_CACHE_PATH.exists():
        print('Loading supervised head from cache:', SUP_CACHE_PATH)
        return joblib.load(SUP_CACHE_PATH)

    print('Training supervised head  (USE_PROPAGATED_SUP_LABELS =',
          USE_PROPAGATED_SUP_LABELS, ')...')
    sup_terms  = build_supervised_terms(train_terms, SUP_TERMS_PER_ONTOLOGY)
    prot_terms = {pid: prot_to_terms.get(pid, []) for pid in train_ids}
    models = {}
    mlbs   = {}

    for o in ['MFO', 'BPO', 'CCO']:
        classes = sup_terms[o]
        mlb     = MultiLabelBinarizer(classes=classes, sparse_output=True)

        y_lists = []
        for pid in train_ids:
            direct = [t for t in prot_terms[pid] if term_to_ont.get(t) == o]
            if USE_PROPAGATED_SUP_LABELS:
                # Expand to all GO ancestors within this ontology.
                # Symmetric with how true labels are evaluated (CAFA protocol).
                expanded = set(direct)
                for t in direct:
                    expanded.update(ancestors_map.get(t, set()))
                terms = [t for t in expanded if term_to_ont.get(t) == o]
            else:
                terms = direct
            y_lists.append(terms)

        Y = mlb.fit_transform(y_lists)

        base = SGDClassifier(
            loss='log_loss', alpha=1e-5, max_iter=5, tol=1e-3, random_state=SEED,
        )
        clf = OneVsRestClassifier(base, n_jobs=-1)
        clf.fit(train_emb, Y)
        models[o] = clf
        mlbs[o]   = mlb
        print(f'  {o}: {Y.shape[1]} labels, density {Y.nnz / (Y.shape[0] * Y.shape[1]):.4f}')

    payload = {'models': models, 'mlbs': mlbs, 'sup_terms': sup_terms}
    joblib.dump(payload, SUP_CACHE_PATH)
    print('Saved supervised head cache:', SUP_CACHE_PATH)
    return payload


def supervised_scores_legacy(payload, emb: np.ndarray) -> list[dict]:
    """Returns list[dict(term->prob)] for each row in emb."""
    if payload is None:
        return [dict() for _ in range(emb.shape[0])]

    models = payload['models']
    mlbs   = payload['mlbs']
    out    = [defaultdict(float) for _ in range(emb.shape[0])]

    for o in ['MFO', 'BPO', 'CCO']:
        clf    = models[o]
        mlb    = mlbs[o]
        dec    = clf.decision_function(emb)
        if dec.ndim == 1:
            dec = dec[:, None]
        probs   = _sigmoid(dec)
        classes = mlb.classes_
        for i in range(emb.shape[0]):
            top_idx = np.argsort(-probs[i])[:128]
            for j in top_idx:
                out[i][classes[j]] = float(probs[i, j])

    return [dict(d) for d in out]


sup_payload = fit_or_load_supervised_head(
    cache_path=SUP_CACHE_PATH,
    train_emb=train_emb,
    train_ids=train_ids,
    train_terms_df=train_terms,
    prot_to_terms=prot_to_terms,
    term_to_ont=term_to_ont,
    ancestors_map=ancestors_map,
    ia_map=ia_map,
    n_per_ont=SUP_TERMS_PER_ONTOLOGY,
    hidden_dims=MLP_HIDDEN_DIMS,
    device=DEVICE,
    seed=SEED,
    use_propagated_labels=USE_PROPAGATED_SUP_LABELS,
    epochs=MLP_EPOCHS,
)


Train MLP MFO:   0%|          | 0/6 [00:00<?, ?it/s]

  MFO: 276 labels, density 0.0894


Train MLP BPO:   0%|          | 0/6 [00:00<?, ?it/s]

  BPO: 956 labels, density 0.0101


Train MLP CCO:   0%|          | 0/6 [00:00<?, ?it/s]

  CCO: 246 labels, density 0.0998
Saved supervised head cache: cafa_work/sup_head_facebook_esm2_t33_650M_UR50D_last3_concat_homology_80_20_dev500_seed472_split20_sup_mlp_mfo1024_bpo2048_cco1024_ep6_mfo768_bpo3072_cco768_proplbl.joblib


## Step 4 — Ensemble multiple backbones (ESM2 + ProtT5)
If `USE_PROTT5=True`, we compute ProtT5 embeddings and average prediction scores across backbones. This is optional because ProtT5 is very large.

In [ ]:
def average_score_dicts(dicts: list[dict]) -> dict:
    if not dicts:
        return {}
    out = defaultdict(float)
    w = 1.0 / len(dicts)
    for d in dicts:
        for k, v in d.items():
            out[k] += w * float(v)
    return dict(out)


# -----------------------------------------------------------------------
# Sequence-similarity label transfer
# -----------------------------------------------------------------------
def seq_sim_label_transfer_legacy(ids: list[str], hits_dict: dict) -> list[dict]:
    """
    For each id in ids look up its precomputed top-K homologues in
    hits_dict = {pid: [(train_pid, norm_score), ...]}.
    Transfers GO labels from training proteins weighted by similarity
    score * sqrt(IA) (mirrors the kNN IA-weighting logic).

    Returns list[dict(term -> score)], one entry per id.
    """
    result = []
    for pid in ids:
        hits = hits_dict.get(pid, [])
        if not hits:
            result.append({})
            continue

        scores = defaultdict(float)
        total_weight = sum(s for _, s in hits) + 1e-12

        for train_pid, sim in hits:
            w = sim / total_weight
            for term in prot_to_terms.get(train_pid, []):
                ia_w = sqrt_ia_map.get(term, 1.0) if USE_IA_WEIGHTED_KNN else 1.0
                scores[term] += w * ia_w

        total_score = sum(scores.values()) + 1e-12
        result.append({t: s / total_score for t, s in scores.items()})

    return result


def predict_raw_scores_esm2(ids: list[str], emb: np.ndarray, *, k: int) -> list[dict]:
    # kNN (+ optional taxon blend) + optional supervised head blend, no propagation yet
    knn_raw = predict_knn_raw_scores_with_taxon(ids, emb, k=k)

    if not USE_SUPERVISED_HEAD:
        return knn_raw

    sup_raw = supervised_scores(sup_payload, emb, device=DEVICE)
    blended = []
    for a, b in zip(sup_raw, knn_raw):
        blended.append(blend_score_dicts(a, b, SUP_BLEND_WEIGHT))
    return blended


def predict_raw_scores_prott5(ids: list[str], emb: np.ndarray, *, k: int) -> list[dict]:
    # ProtT5 branch: kNN only (supervised head optional but off by default in this notebook)
    # Build global index lazily once
    global prott5_index, prott5_train_term_lists

    if prott5_train_emb is None:
        raise RuntimeError('USE_PROTT5=True but ProtT5 embeddings are not available.')

    if 'prott5_index' not in globals():
        prott5_train_term_lists = train_term_lists
        prott5_index = build_faiss_index(prott5_train_emb)

    raw = knn_label_transfer(
        query_emb=emb,
        train_emb=prott5_train_emb,
        train_term_lists=prott5_train_term_lists,
        index=prott5_index,
        k=min(k, prott5_train_emb.shape[0]),
        top_terms_pre_prop=TOP_TERMS_PER_PROTEIN_BEFORE_PROP,
    )
    return raw


def predict_scores(ids: list[str], *, esm2_emb: np.ndarray, prott5_emb: np.ndarray | None, k: int) -> list[dict]:
    knn_branch = predict_knn_raw_scores_with_taxon(ids, esm2_emb, k=k)
    if USE_PROTT5 and prott5_emb is not None:
        raw_t5 = predict_raw_scores_prott5(ids, prott5_emb, k=k)
        knn_branch = [average_score_dicts([a, b]) for a, b in zip(knn_branch, raw_t5)]

    sup_branch = supervised_scores(sup_payload, esm2_emb, device=DEVICE) if USE_SUPERVISED_HEAD else [dict() for _ in ids]

    seq_branch = [dict() for _ in ids]
    seq_meta_by_pid = {pid: {'seq_confidence': 0.0, 'seq_alpha': SEQ_SIM_BLEND_WEIGHT} for pid in ids}
    if USE_SEQ_SIM and seq_sim_hits:
        seq_branch, seq_meta_by_pid = seq_sim_label_transfer(
            ids,
            seq_sim_hits,
            prot_to_terms=prot_to_terms,
            sqrt_ia_map=sqrt_ia_map,
            use_ia_weighted_knn=USE_IA_WEIGHTED_KNN,
        )

    if FUSION_MODE == 'logistic_stack' and ontology_stackers:
        merged_raw = stack_ontology_scores(
            ids,
            knn_scores=knn_branch,
            sup_scores=sup_branch,
            seq_scores=seq_branch,
            term_to_ont=term_to_ont,
            stackers=ontology_stackers,
            ia_map=ia_map,
            depth_map=term_depth_map,
            seq_meta_by_pid=seq_meta_by_pid,
            taxon_meta_by_pid=taxon_meta_by_pid,
        )
    else:
        merged_raw = []
        for pid, knn_row, sup_row, seq_row in zip(ids, knn_branch, sup_branch, seq_branch):
            row = blend_score_dicts(sup_row, knn_row, SUP_BLEND_WEIGHT) if sup_row else dict(knn_row)
            if seq_row:
                row = blend_score_dicts(seq_row, row, seq_meta_by_pid.get(pid, {}).get('seq_alpha', SEQ_SIM_BLEND_WEIGHT))
            merged_raw.append(row)

    if CALIBRATION_MODE != 'none' and ontology_calibrators:
        merged_raw = apply_ontology_calibrators(
            merged_raw,
            term_to_ont=term_to_ont,
            calibrators=ontology_calibrators,
        )

    return [postprocess_scores(d) for d in merged_raw]


In [ ]:
term_depth_map = build_term_depth_map(ancestors_map, term_to_ont)
taxon_meta_by_pid = {
    pid: {'taxon_alpha': float(taxon_bayesian_alpha.get(int(test_tax_map.get(pid, -1)), 0.0))}
    for pid in test_ids
}
val_taxon_meta_by_pid = {
    pid: {'taxon_alpha': float(taxon_bayesian_alpha.get(int(train_tax_map.get(pid, -1)), 0.0))}
    for pid in val_ids
}
ontology_stackers = {}
ontology_calibrators = {}

fit_emb = train_emb[fit_idx]
val_emb = train_emb[val_idx]
fit_term_lists = [prot_to_terms.get(pid, []) for pid in fit_ids]
fit_index = build_faiss_index(fit_emb)

knn_val_raw = knn_label_transfer(
    query_emb=val_emb,
    train_emb=fit_emb,
    train_term_lists=fit_term_lists,
    index=fit_index,
    k=min(K_NEIGHBORS, len(fit_ids)),
    top_terms_pre_prop=TOP_TERMS_PER_PROTEIN_BEFORE_PROP,
)

INNER_SUP_CACHE_PATH = WORK_DIR / f'inner_sup_head_{_BACKBONE}_{_SPLIT_TAG}_{_SUP_TAG}.joblib'
inner_sup_payload = fit_or_load_supervised_head(
    cache_path=INNER_SUP_CACHE_PATH,
    train_emb=fit_emb,
    train_ids=fit_ids,
    train_terms_df=train_terms,
    prot_to_terms=prot_to_terms,
    term_to_ont=term_to_ont,
    ancestors_map=ancestors_map,
    ia_map=ia_map,
    n_per_ont=SUP_TERMS_PER_ONTOLOGY,
    hidden_dims=MLP_HIDDEN_DIMS,
    device=DEVICE,
    seed=SEED,
    use_propagated_labels=USE_PROPAGATED_SUP_LABELS,
    epochs=max(3, MLP_EPOCHS - 2),
)
sup_val_raw = supervised_scores(inner_sup_payload, val_emb, device=DEVICE)

INNER_SEQ_SIM_CACHE_PATH = WORK_DIR / f"inner_seq_sim_hits_{_SPLIT_TAG}_top{SEQ_SIM_TOP_K}_e{str(SEQ_SIM_EVALUE).replace('.', 'p')}.pkl"
if INNER_SEQ_SIM_CACHE_PATH.exists():
    with open(INNER_SEQ_SIM_CACHE_PATH, 'rb') as fh:
        inner_seq_hits = pickle.load(fh)
else:
    inner_seq_hits = compute_seq_sim_hits(
        train_ids=fit_ids,
        test_ids=val_ids,
        train_seqs=train_seqs,
        test_seqs=train_seqs,
        top_k=SEQ_SIM_TOP_K,
        evalue=SEQ_SIM_EVALUE,
        diamond_path=DIAMOND_PATH,
    )
    with open(INNER_SEQ_SIM_CACHE_PATH, 'wb') as fh:
        pickle.dump(inner_seq_hits, fh, protocol=4)
seq_val_raw, seq_meta_val = seq_sim_label_transfer(
    val_ids,
    inner_seq_hits,
    prot_to_terms=prot_to_terms,
    sqrt_ia_map=sqrt_ia_map,
    use_ia_weighted_knn=USE_IA_WEIGHTED_KNN,
)

true_val_prop = {}
for pid in val_ids:
    direct = set(prot_to_terms.get(pid, []))
    expanded = set(direct)
    for term in direct:
        expanded.update(ancestors_map.get(term, set()))
    true_val_prop[pid] = expanded & ia_terms

if FUSION_MODE == 'logistic_stack':
    ontology_stackers = fit_ontology_stackers(
        val_ids,
        knn_scores=knn_val_raw,
        sup_scores=sup_val_raw,
        seq_scores=seq_val_raw,
        true_by_prot=true_val_prop,
        term_to_ont=term_to_ont,
        ia_map=ia_map,
        depth_map=term_depth_map,
        seq_meta_by_pid=seq_meta_val,
        taxon_meta_by_pid=val_taxon_meta_by_pid,
    )

if FUSION_MODE == 'logistic_stack' and ontology_stackers:
    merged_val_raw = stack_ontology_scores(
        val_ids,
        knn_scores=knn_val_raw,
        sup_scores=sup_val_raw,
        seq_scores=seq_val_raw,
        term_to_ont=term_to_ont,
        stackers=ontology_stackers,
        ia_map=ia_map,
        depth_map=term_depth_map,
        seq_meta_by_pid=seq_meta_val,
        taxon_meta_by_pid=val_taxon_meta_by_pid,
    )
else:
    merged_val_raw = []
    for pid, knn_row, sup_row, seq_row in zip(val_ids, knn_val_raw, sup_val_raw, seq_val_raw):
        row = blend_score_dicts(sup_row, knn_row, SUP_BLEND_WEIGHT) if sup_row else dict(knn_row)
        if seq_row:
            row = blend_score_dicts(seq_row, row, seq_meta_val.get(pid, {}).get('seq_alpha', SEQ_SIM_BLEND_WEIGHT))
        merged_val_raw.append(row)

if CALIBRATION_MODE != 'none':
    pred_val_map = {pid: row for pid, row in zip(val_ids, merged_val_raw)}
    ontology_calibrators = fit_ontology_calibrators(
        pred_val_map,
        true_val_prop,
        term_to_ont=term_to_ont,
        mode=CALIBRATION_MODE,
    )

print(f'Prepared stackers: {sorted(ontology_stackers)} | calibrators: {sorted(ontology_calibrators)}')


Train MLP MFO:   0%|          | 0/4 [00:00<?, ?it/s]

  MFO: 270 labels, density 0.0918


Train MLP BPO:   0%|          | 0/4 [00:00<?, ?it/s]

  BPO: 915 labels, density 0.0105


Train MLP CCO:   0%|          | 0/4 [00:00<?, ?it/s]

  CCO: 239 labels, density 0.1017
Saved supervised head cache: cafa_work/inner_sup_head_facebook_esm2_t33_650M_UR50D_last3_concat_homology_80_20_dev500_seed472_split20_sup_mlp_mfo1024_bpo2048_cco1024_ep6_mfo768_bpo3072_cco768_proplbl.joblib
  blastp not found; skipping...
  Using k-mer cosine similarity (no DIAMOND/BLAST found)
  Building k-mer matrix for 378 train + 22 test proteins ...
  k-mer: 220 hits for 22/22 query proteins
Prepared stackers: ['BPO', 'CCO', 'MFO'] | calibrators: ['BPO', 'CCO', 'MFO']


In [ ]:
# Optional: sanity check on a small batch of held-out test proteins
sample_ids = test_ids[:4]
idxs = np.array([test_ids.index(pid) for pid in sample_ids])

sample_esm2 = test_emb[idxs]
sample_t5   = prott5_test_emb[idxs] if (USE_PROTT5 and prott5_test_emb is not None) else None

preds = predict_scores(sample_ids, esm2_emb=sample_esm2, prott5_emb=sample_t5, k=K_NEIGHBORS)
for pid, d in zip(sample_ids, preds):
    top5_pred = sorted(d.items(), key=lambda x: x[1], reverse=True)[:5]
    true_terms = set(prot_to_terms.get(pid, []))
    hits        = sum(1 for t, _ in top5_pred if t in true_terms)
    print(f'{pid}  taxon={test_tax_map.get(pid, -1)}  top5={top5_pred}  true_hits={hits}/{len(top5_pred)}')

A0A0R4ILB2  taxon=7955  top5=[('GO:0016491', 0.2952178965682571), ('GO:0016175', 0.2952178965682571), ('GO:0004491', 0.2952178965682571), ('GO:0051864', 0.2952178965682571), ('GO:0047075', 0.2952178965682571)]  true_hits=0/5
A2R6H0  taxon=425011  top5=[('GO:0016712', 0.2893998802077387), ('GO:0016491', 0.2893998802077387), ('GO:0003958', 0.2893998802077387), ('GO:0004491', 0.2893998802077387), ('GO:0008726', 0.2893998802077387)]  true_hits=0/5
A3KGV1  taxon=10090  top5=[('GO:0001968', 0.24230467429389962), ('GO:0030332', 0.23867666151951974), ('GO:0051018', 0.22844107243441034), ('GO:0034236', 0.22844107243441034), ('GO:0034237', 0.22844107243441034)]  true_hits=0/5
A4D998  taxon=3702  top5=[('GO:0001968', 0.24793409510894582), ('GO:0051018', 0.23468650026734364), ('GO:0034236', 0.23468650026734364), ('GO:0034237', 0.23468650026734364), ('GO:0008013', 0.22453038705679826)]  true_hits=0/5


## Evaluate on the held-out test set
We run the full prediction pipeline on the **20 % held-out test proteins** and compare against their true GO-term labels.

Metrics reported per ontology (MFO, BPO, CCO) and overall:
- **IA-weighted Precision (wP)** — how accurate are the predicted terms?
- **IA-weighted Recall (wR)** — how many true terms are recovered?
- **IA-weighted F-score (wF)** — harmonic mean, evaluated at the best threshold (max-F over a threshold sweep)


In [ ]:
import pickle

ia_map = dict(zip(ia['term'].values, ia['ia'].values))


# ------------------------------------------------------------------
# True-label propagation (CAFA protocol)
# ------------------------------------------------------------------
def propagate_true_labels(direct_terms: set) -> set:
    expanded = set(direct_terms)
    for t in direct_terms:
        expanded.update(ancestors_map.get(t, set()))
    return expanded & ia_terms


# ------------------------------------------------------------------
# IA-filter + max-normalise predictions
# ------------------------------------------------------------------
def clean_pred_scores(d: dict) -> dict:
    d = {t: s for t, s in d.items() if t in ia_terms}
    if USE_PER_PROTEIN_MAX_NORMALIZATION and d:
        max_s = max(d.values())
        if max_s > 0:
            return {t: s / max_s for t, s in d.items()}
    return d


# ------------------------------------------------------------------
# Threshold sweep
# ------------------------------------------------------------------
def make_thresholds():
    return np.unique(np.concatenate([
        np.geomspace(0.001, 0.1, 50),
        np.linspace(0.1, 0.95, 50),
    ]))


# ------------------------------------------------------------------
# Pre-compute top-1 BPO term per protein for the coverage floor
# ------------------------------------------------------------------
def get_top_bpo_terms(pred_by_prot: dict) -> dict:
    """Returns {pid: top_bpo_term_or_None} for every protein."""
    result = {}
    for pid, scores in pred_by_prot.items():
        bpo_items = [(t, s) for t, s in scores.items() if term_to_ont.get(t) == 'BPO']
        result[pid] = max(bpo_items, key=lambda x: x[1])[0] if bpo_items else None
    return result


# ------------------------------------------------------------------
# IA-weighted per-protein P / R / F
# ------------------------------------------------------------------
def weighted_prf_at_threshold(true_terms, pred_scores, threshold, ont_filter=None):
    pred_terms = {t for t, s in pred_scores.items() if s >= threshold}
    if ont_filter is not None:
        true_terms = {t for t in true_terms if term_to_ont.get(t) == ont_filter}
        pred_terms = {t for t in pred_terms if term_to_ont.get(t) == ont_filter}
    if not pred_terms:
        return 0.0, 0.0, 0.0
    inter    = true_terms & pred_terms
    wp_num   = sum(ia_map.get(t, 0.0) for t in inter)
    wp_denom = sum(ia_map.get(t, 0.0) for t in pred_terms) + 1e-12
    wr_denom = sum(ia_map.get(t, 0.0) for t in true_terms) + 1e-12
    wP = wp_num / wp_denom
    wR = wp_num / wr_denom
    wF = (2 * wP * wR / (wP + wR)) if (wP + wR) > 0 else 0.0
    return wP, wR, wF


# ------------------------------------------------------------------
# Standard unweighted per-protein P / R / F1
# ------------------------------------------------------------------
def unweighted_prf_at_threshold(true_terms, pred_scores, threshold, ont_filter=None):
    pred_terms = {t for t, s in pred_scores.items() if s >= threshold}
    if ont_filter is not None:
        true_terms = {t for t in true_terms if term_to_ont.get(t) == ont_filter}
        pred_terms = {t for t in pred_terms if term_to_ont.get(t) == ont_filter}
    if not pred_terms:
        return 0.0, 0.0, 0.0
    inter = true_terms & pred_terms
    P = len(inter) / len(pred_terms)
    R = len(inter) / (len(true_terms) + 1e-12)
    F = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0
    return P, R, F


# ------------------------------------------------------------------
# Threshold sweep with optional per-protein coverage floor.
#
# Improvement #6 — BPO coverage floor (correct implementation):
#   At any threshold θ, if a BPO-evaluable protein has ZERO BPO terms
#   above θ, we guarantee its single highest-scoring BPO term is still
#   counted as a prediction.  This is evaluated jointly with θ in the
#   sweep, so the optimal threshold now accounts for the floor.
#   Effect: forces non-zero BPO recall for all covered proteins without
#   flooding every protein with uncertain low-score terms.
# ------------------------------------------------------------------
def evaluate_predictions(true_by_prot, pred_by_prot, ontology=None,
                         thresholds=None, top_ont_terms=None):
    """top_ont_terms: optional dict {pid: term} — coverage floor fallback."""
    if thresholds is None:
        thresholds = make_thresholds()
    thresholds = np.asarray(thresholds, dtype=float)
    default_th = float(thresholds[0]) if thresholds.size else 0.5
    if not true_by_prot:
        return 0.0, default_th, 0.0, 0.0
    best = (0.0, default_th, 0.0, 0.0)
    for th in thresholds:
        ps, rs, fs = [], [], []
        for pid in true_by_prot:
            scores = pred_by_prot.get(pid, {})
            # Apply coverage floor: if no ontology predictions above th, add top-1
            if top_ont_terms is not None and ontology is not None:
                ont_above = any(
                    s >= th and term_to_ont.get(t) == ontology
                    for t, s in scores.items()
                )
                if not ont_above and top_ont_terms.get(pid) is not None:
                    scores = dict(scores)
                    scores[top_ont_terms[pid]] = th   # floor: just at threshold
            wP, wR, wF = weighted_prf_at_threshold(
                true_by_prot[pid], scores, th, ont_filter=ontology)
            ps.append(wP); rs.append(wR); fs.append(wF)
        mf = float(np.mean(fs)) if fs else 0.0
        if mf > best[0]:
            best = (mf, float(th), float(np.mean(ps)), float(np.mean(rs)))
    return best


# ------------------------------------------------------------------
# Load or compute test predictions
# ------------------------------------------------------------------
if TEST_PREDS_CACHE_PATH.exists():
    print(f'Loading cached predictions: {TEST_PREDS_CACHE_PATH.name}')
    with open(TEST_PREDS_CACHE_PATH, 'rb') as _f:
        test_preds = pickle.load(_f)
    print(f'  {len(test_preds):,} proteins loaded.')
else:
    print(f'Predicting {len(test_ids):,} held-out proteins ...')
    test_preds = {}
    EVAL_BATCH = 128
    for i in tqdm(range(0, len(test_ids), EVAL_BATCH), desc='Test prediction'):
        b_ids  = test_ids[i:i + EVAL_BATCH]
        b_esm2 = test_emb[i:i + EVAL_BATCH]
        b_t5   = (prott5_test_emb[i:i + EVAL_BATCH]
                  if (USE_PROTT5 and prott5_test_emb is not None) else None)
        b_preds = predict_scores(b_ids, esm2_emb=b_esm2, prott5_emb=b_t5,
                                  k=K_NEIGHBORS)
        for pid, d in zip(b_ids, b_preds):
            test_preds[pid] = d
    with open(TEST_PREDS_CACHE_PATH, 'wb') as _f:
        pickle.dump(test_preds, _f, protocol=pickle.HIGHEST_PROTOCOL)
    sz = TEST_PREDS_CACHE_PATH.stat().st_size // 1024
    print(f'Saved -> {TEST_PREDS_CACHE_PATH.name} ({sz:,} KB)')

# IA-filter + max-normalise
clean_preds = {pid: clean_pred_scores(d) for pid, d in test_preds.items()}

# Pre-compute top-1 BPO term per protein (used for coverage floor in sweep)
top_bpo_terms = get_top_bpo_terms(clean_preds) if BPO_COVERAGE_FLOOR else {}
bpo_covered   = sum(1 for t in top_bpo_terms.values() if t is not None)
print(f'BPO signal present in {bpo_covered:,} / {len(clean_preds):,} proteins'
      f'  (coverage floor {"ON" if BPO_COVERAGE_FLOOR else "OFF"})')

# ------------------------------------------------------------------
# Propagate true labels
# ------------------------------------------------------------------
print('Propagating true labels through the GO DAG ...', end=' ')
true_test_prop = {pid: propagate_true_labels(set(prot_to_terms.get(pid, [])))
                  for pid in test_ids}
print('done.')

# ------------------------------------------------------------------
# Evaluate per ontology
# ------------------------------------------------------------------
all_ont_f = []
results   = {}

for ont in ['MFO', 'BPO', 'CCO']:
    ont_pids = [pid for pid in test_ids
                if any(term_to_ont.get(t) == ont for t in true_test_prop[pid])]
    true_sub = {pid: true_test_prop[pid] for pid in ont_pids}
    pred_sub = {pid: clean_preds[pid]    for pid in ont_pids}

    # Use coverage floor for BPO sweep
    floor_map = top_bpo_terms if (ont == 'BPO' and BPO_COVERAGE_FLOOR) else None

    best_f, best_th, best_p, best_r = evaluate_predictions(
        true_sub, pred_sub, ontology=ont, top_ont_terms=floor_map)
    all_ont_f.append(best_f)

    # Standard metrics at best_th, also with coverage floor applied
    std_ps, std_rs, std_fs, coverages, n_preds_list = [], [], [], [], []
    for pid in ont_pids:
        scores = clean_preds.get(pid, {})
        # Apply floor for BPO metric reporting
        if ont == 'BPO' and BPO_COVERAGE_FLOOR:
            ont_above = any(s >= best_th and term_to_ont.get(t) == 'BPO'
                            for t, s in scores.items())
            if not ont_above and top_bpo_terms.get(pid) is not None:
                scores = dict(scores)
                scores[top_bpo_terms[pid]] = best_th
        sp, sr, sf = unweighted_prf_at_threshold(
            true_test_prop[pid], scores, best_th, ont_filter=ont)
        std_ps.append(sp); std_rs.append(sr); std_fs.append(sf)
        n_ont = sum(1 for t, s in scores.items()
                    if s >= best_th and term_to_ont.get(t) == ont)
        coverages.append(1 if n_ont > 0 else 0)
        n_preds_list.append(n_ont)

    results[ont] = {
        'n':         len(ont_pids),
        'wF':        best_f,
        'wP':        best_p,
        'wR':        best_r,
        'th':        best_th,
        'P':         float(np.mean(std_ps)),
        'R':         float(np.mean(std_rs)),
        'F1':        float(np.mean(std_fs)),
        'cov':       float(np.mean(coverages)) * 100,
        'avg_preds': float(np.mean(n_preds_list)),
    }

mean_f   = float(np.mean(all_ont_f))
mean_wP  = float(np.mean([r['wP']  for r in results.values()]))
mean_wR  = float(np.mean([r['wR']  for r in results.values()]))
mean_P   = float(np.mean([r['P']   for r in results.values()]))
mean_R   = float(np.mean([r['R']   for r in results.values()]))
mean_F1  = float(np.mean([r['F1']  for r in results.values()]))
mean_cov = float(np.mean([r['cov'] for r in results.values()]))

# ------------------------------------------------------------------
# Print results
# ------------------------------------------------------------------
W    = 80
sep  = '=' * W
thin = '-' * W

print()
print(sep)
print(f'  CAFA-6 Evaluation  -  {len(test_ids):,} held-out proteins'
      f'  ({_ALGO_TAG})')
print(sep)

print()
print('  [ IA-Weighted metrics  -  official CAFA protocol ]')
print(f'  {"Ontology":>8}  {"#Prot":>6}  {"Threshold":>9}'
      f'  {"wF (best)":>10}  {"wPrecision":>10}  {"wRecall":>9}')
print(f'  {thin}')
for ont in ['MFO', 'BPO', 'CCO']:
    r = results[ont]
    print(f'  {ont:>8}  {r["n"]:>6,}  {r["th"]:>9.4f}'
          f'  {r["wF"]:>10.4f}  {r["wP"]:>10.4f}  {r["wR"]:>9.4f}')
print(f'  {thin}')
print(f'  {"Mean":>8}  {"":>6}  {"":>9}'
      f'  {mean_f:>10.4f}  {mean_wP:>10.4f}  {mean_wR:>9.4f}')

print()
print('  [ Standard (unweighted) metrics  @  IA-optimal threshold ]')
print(f'  {"Ontology":>8}  {"#Prot":>6}  {"Threshold":>9}'
      f'  {"F1":>7}  {"Precision":>9}  {"Recall":>7}'
      f'  {"Coverage":>9}  {"AvgPreds":>8}')
print(f'  {thin}')
for ont in ['MFO', 'BPO', 'CCO']:
    r = results[ont]
    print(f'  {ont:>8}  {r["n"]:>6,}  {r["th"]:>9.4f}'
          f'  {r["F1"]:>7.4f}  {r["P"]:>9.4f}  {r["R"]:>7.4f}'
          f'  {r["cov"]:>8.1f}%  {r["avg_preds"]:>8.1f}')
print(f'  {thin}')
print(f'  {"Mean":>8}  {"":>6}  {"":>9}'
      f'  {mean_F1:>7.4f}  {mean_P:>9.4f}  {mean_R:>7.4f}'
      f'  {mean_cov:>8.1f}%')

print()
print(sep)
print()
print('  Metric guide')
print('  ' + '-' * 70)
print('  wF / wP / wR   CAFA official IA-weighted F / Precision / Recall')
print('                 Specific (high-IA) GO terms count more than generic ones.')
print('  F1 / P / R     Unweighted macro-avg F1 / Precision / Recall per protein')
print('                 Every correctly predicted GO term counts equally.')
print('  Coverage       % proteins with >= 1 predicted term above threshold')
print('  AvgPreds       Average number of GO terms predicted per protein')
print(sep)


Loading cached predictions: test_preds_facebook_esm2_t33_650M_UR50D_last3_concat_homology_80_20_dev500_seed472_split20_k500_mf3_rw1_iaw1_ss1_dta1_calplatt_fuslogistic_stack_sup_mlp_mfo1024_bpo2048_cco1024_ep6_mfo768_bpo3072_cco768_proplbl.pkl
  100 proteins loaded.
BPO signal present in 0 / 100 proteins  (coverage floor ON)
Propagating true labels through the GO DAG ... done.

  CAFA-6 Evaluation  -  100 held-out proteins  (k500_mf3_rw1_iaw1_ss1_dta1_calplatt_fuslogistic_stack)

  [ IA-Weighted metrics  -  official CAFA protocol ]
  Ontology   #Prot  Threshold   wF (best)  wPrecision    wRecall
  --------------------------------------------------------------------------------
       MFO      75     0.1867      0.5113      0.5088     0.5749
       BPO      77     0.0010      0.0000      0.0000     0.0000
       CCO      76     0.0010      0.0701      0.2020     0.0784
  --------------------------------------------------------------------------------
      Mean                         0.

### CAFA-6 notebook: improvement log

| Session | Change | Effect |
|---------|--------|--------|
| 1 | 80:20 stratified train/test split | Proper held-out test |
| 1 | Ontology remap bugfix (C/F/P → CCO/MFO/BPO) | All term counts fixed |
| 2 | True label propagation through GO DAG | Mean wF: ~0.005 → 0.337 |
| 2 | IA-term filtering + per-protein max-normalise | CAFA-compliant eval |
| 2 | Finer threshold sweep (log + linear) | More stable wF |
| 3 | Rank-weighted kNN (1/(rank+1)) | Smoother voting |
| 3 | IA-weighted kNN term scoring (×sqrt(IA)) | Aligns scoring with metric |
| 3 | Min-frequency term filter (≥3 proteins) | Removes label noise |
| 3 | K=400→500 neighbours | Better BPO coverage |
| 3 | Per-ontology supervised terms (BPO: 3072) | Better BPO label coverage |
| 3 | SUP_BLEND_WEIGHT 0.5→0.30 | More kNN weight |
| 4 | **Propagated training labels in sup head** | Head learns ancestor calibration |
| 4 | **BPO coverage floor in threshold sweep** | Guarantees ≥1 BPO prediction |

**Scores at session 3 end (old sup head, no floor):** Mean wF = 0.3804
- MFO: 0.6064 | BPO: 0.0950 | CCO: 0.4400

**Scores after session 4 (#6 floor only, #3 pending re-prediction):** Mean wF = 0.3805
- MFO: 0.6064 | BPO: 0.0953 | CCO: 0.4400

**Note:** Improvement #3 (propagated sup-head labels) takes full effect on the next
predictions run — the new cache `test_preds_..._proplbl.pkl` will be built
automatically since the old cache name no longer matches.

**Remaining high-impact ideas (ordered by expected gain):**
1. ESM2 650M backbone (`esm2_t33_650M_UR50D`) — +8–15 wF, needs one re-embedding
2. DIAMOND/BLAST sequence-similarity signal — especially strong for BPO
3. 2-layer MLP supervised head instead of linear SGD
4. Iterative label propagation on protein similarity graph
